In [2]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/magnetic")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 960, 540



FPS = 24



FRAMES = 180







BG = (2, 7, 13, 255)







CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)



GREEN = (90, 255, 170)



PURPLE = (190, 120, 255)



WHITE = (245, 250, 255)







def load_font(size):



    candidates = [



        "/System/Library/Fonts/Menlo.ttc",



        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",



        "/Library/Fonts/Arial.ttf",



        "DejaVuSansMono.ttf",



    ]



    for path in candidates:



        try:



            return ImageFont.truetype(path, size)



        except Exception:



            pass



    return ImageFont.load_default()







FONT = load_font(22)



FONT_SMALL = load_font(14)



FONT_TINY = load_font(11)











def dipole_field(x, y, phase):



    cx = W * 0.48



    cy = H * 0.50







    dx = x - cx



    dy = y - cy







    tilt = 0.45 * np.sin(phase * 0.45)







    xr = dx * np.cos(tilt) - dy * np.sin(tilt)



    yr = dx * np.sin(tilt) + dy * np.cos(tilt)







    r2 = xr * xr + yr * yr + 1200



    r = np.sqrt(r2)







    bx = (3 * xr * yr) / (r**5)



    by = (2 * yr * yr - xr * xr) / (r**5)







    scale = 2.8e8







    return bx * scale, by * scale











def trace_field_line(x0, y0, phase, step=4, n=180, direction=1):



    pts = []







    x = x0



    y = y0







    for _ in range(n):



        bx, by = dipole_field(x, y, phase)







        norm = np.hypot(bx, by)







        if norm < 1e-6:



            break







        bx = direction * bx / norm



        by = direction * by / norm







        x += bx * step



        y += by * step







        pts.append((x, y))







        if x < -40 or x > W + 40 or y < -40 or y > H + 40:



            break







    return pts











def render_frame(i):



    phase = 2 * np.pi * i / FRAMES







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    cx = W * 0.48



    cy = H * 0.50







    # Header



    d.text((32, 10), "MAGNETIC FIELD MAPPER", font=FONT_SMALL, fill=(*CYAN2, 190))



    d.line((28, 34, 205, 34), fill=(*CYAN, 170), width=2)



    d.line((205, 34, 228, 48), fill=(*CYAN, 170), width=2)



    d.line((228, 48, W - 42, 48), fill=(*CYAN, 75), width=1)







    # Background grid



    for x in range(80, W, 80):



        d.line((x, 72, x, H - 50), fill=(*CYAN, 12), width=1)







    for y in range(90, H - 40, 60):



        d.line((40, y, W - 40, y), fill=(*CYAN, 12), width=1)







    # Star / compact object



    for r, a in [(64, 28), (44, 55), (24, 140)]:



        d.ellipse(



            [cx-r, cy-r, cx+r, cy+r],



            outline=(*WHITE, a),



            width=2,



        )







    d.ellipse(



        [cx-12, cy-12, cx+12, cy+12],



        fill=(*WHITE, 245),



    )







    # Magnetic poles



    tilt = 0.45 * np.sin(phase * 0.45)







    px1 = cx + np.sin(tilt) * 58



    py1 = cy - np.cos(tilt) * 58







    px2 = cx - np.sin(tilt) * 58



    py2 = cy + np.cos(tilt) * 58







    pulse = 0.65 + 0.35 * np.sin(phase * 5)







    d.ellipse(



        [px1-8, py1-8, px1+8, py1+8],



        fill=(*GREEN, int(180 * pulse)),



    )







    d.ellipse(



        [px2-8, py2-8, px2+8, py2+8],



        fill=(*PURPLE, int(180 * pulse)),



    )







    # Field lines



    seeds = []







    for a in np.linspace(-1.6, 1.6, 24):



        seeds.append((cx + a * 34, cy - 64))



        seeds.append((cx + a * 34, cy + 64))











    for idx, (sx, sy) in enumerate(seeds):







        pts_back = trace_field_line(sx, sy, phase, direction=-1)



        pts_fwd = trace_field_line(sx, sy, phase, direction=1)







        pts = list(reversed(pts_back)) + [(sx, sy)] + pts_fwd







        if len(pts) < 2:



            continue







        alpha = int(40 + 70 * np.sin(phase * 3 + idx) ** 2)







        color = CYAN if idx % 2 == 0 else CYAN2







        # glow



        for p1, p2 in zip(pts[:-1], pts[1:]):



            d.line([p1, p2], fill=(*color, alpha // 2), width=3)







        # main line



        for p1, p2 in zip(pts[:-1], pts[1:]):



            d.line([p1, p2], fill=(*color, alpha), width=1)







    # Reconnection arcs



    for k in range(4):



        arc_phase = phase * 1.5 + k * 1.3







        x1 = cx + np.cos(arc_phase) * 120



        y1 = cy + np.sin(arc_phase * 1.2) * 60







        x2 = x1 + 45 * np.sin(phase * 3 + k)



        y2 = y1 + 18 * np.cos(phase * 4 + k)







        d.line(



            [x1, y1, x2, y2],



            fill=(*GREEN, 120),



            width=1,



        )







    # Vector probes



    for gx in range(140, 760, 120):



        for gy in range(120, 460, 90):







            bx, by = dipole_field(gx, gy, phase)







            norm = max(np.hypot(bx, by), 1e-5)







            bx /= norm



            by /= norm







            scale = 14







            x2 = gx + bx * scale



            y2 = gy + by * scale







            a = int(50 + 80 * np.sin(phase * 2 + gx * 0.01 + gy * 0.01) ** 2)







            d.line(



                [gx, gy, x2, y2],



                fill=(*CYAN2, a),



                width=1,



            )







            d.ellipse(



                [x2-1, y2-1, x2+1, y2+1],



                fill=(*CYAN2, a),



            )







    # Right telemetry panel



    px, py = 710, 95



    pw, ph = 210, 250







    d.rectangle(



        [px, py, px + pw, py + ph],



        outline=(*CYAN, 100),



        width=1,



    )







    d.line(



        (px, py, px + 80, py),



        fill=(*CYAN2, 190),



        width=3,



    )







    d.text(



        (px + 16, py + 16),



        "FIELD SOLVER",



        font=FONT_SMALL,



        fill=(*CYAN2, 190),



    )







    metrics = [



        ("B-FIELD", f"{42 + 4*np.sin(phase):04.1f} MG"),



        ("TILT", f"{12 + 8*np.sin(phase*0.5):04.1f}°"),



        ("FLUX", f"{0.74 + 0.06*np.cos(phase):.3f}"),



        ("SHEAR", f"{0.18 + 0.04*np.sin(phase*1.7):.3f}"),



        ("LOCK", f"{96 + 3*np.sin(phase*2.2):04.1f}%"),



    ]







    for m, (k, v) in enumerate(metrics):







        yy = py + 56 + m * 36







        d.text(



            (px + 16, yy),



            k,



            font=FONT_TINY,



            fill=(*CYAN, 150),



        )







        d.text(



            (px + 90, yy),



            v,



            font=FONT_SMALL,



            fill=(*CYAN2, 220),



        )







        for t in range(5):



            tx = px + 176 + t * 6







            aa = int(35 + 90 * np.sin(phase * 4 + t + m) ** 2)







            d.line(



                (tx, yy + 8, tx + 3, yy + 8),



                fill=(*CYAN, aa),



                width=1,



            )







    # Footer



    d.text(



        (32, H - 30),



        "MAGNETIC TOPOLOGY RECONSTRUCTION • FIELD LINE LOCK ACTIVE • DIPOLE SOLUTION STABLE",



        font=FONT_TINY,



        fill=(*CYAN, 130),



    )







    # HUD corners



    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)



    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)







    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)



    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)







    glow = img.filter(ImageFilter.GaussianBlur(2.0))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as _np



from vizlib.animation_export import export_animation







_arrays = [_np.array(f.convert("RGB")) for f in frames]



saved = export_animation(_arrays, OUT_DIR, "magnetic_field_mapper", "webm", FPS)



print(f"Saved: {saved}")



Saved: animations/magnetic/magnetic_field_mapper.gif


In [3]:
from pathlib import Path

import numpy as np

from PIL import Image, ImageDraw, ImageFilter



OUT_DIR = Path("media-site/animations/gravity")

OUT_DIR.mkdir(parents=True, exist_ok=True)



W, H = 960, 540

FPS = 24

FRAMES = 168



BG = (2, 7, 13, 255)



CYAN = (90, 240, 255)

CYAN2 = (180, 255, 255)

GREY = (130, 145, 155)

WHITE = (245, 250, 255)





def lens_warp(x, y, phase):

    cx = W * 0.50 + 18 * np.sin(phase * 0.6)

    cy = H * 0.50 + 12 * np.cos(phase * 0.7)



    dx = x - cx

    dy = y - cy



    r2 = dx * dx + dy * dy + 900

    r = np.sqrt(r2)



    strength = 13500 + 1800 * np.sin(phase * 0.8)



    # radial lensing displacement

    shift = strength / r2



    wx = x + dx / r * shift

    wy = y + dy / r * shift



    # weak shear / rotation

    shear = 0.018 * np.sin(phase)

    wx += shear * dy

    wy += shear * dx



    return wx, wy





def draw_polyline(draw, points, fill, width=1):

    for p1, p2 in zip(points[:-1], points[1:]):

        draw.line([p1, p2], fill=fill, width=width)





def render_frame(i):

    phase = 2 * np.pi * i / FRAMES



    img = Image.new("RGBA", (W, H), BG)

    d = ImageDraw.Draw(img)



    cx = W * 0.50 + 18 * np.sin(phase * 0.6)

    cy = H * 0.50 + 12 * np.cos(phase * 0.7)



    # distorted reference grid

    step = 64



    for x in range(-step, W + step, step):

        pts = []



        for y in range(0, H + 1, 8):

            pts.append(lens_warp(x, y, phase))



        draw_polyline(d, pts, (*GREY, 42), width=1)



    for y in range(-step, H + step, step):

        pts = []



        for x in range(0, W + 1, 8):

            pts.append(lens_warp(x, y, phase))



        draw_polyline(d, pts, (*GREY, 38), width=1)



    # Einstein ring

    ring_r = 108 + 5 * np.sin(phase * 1.7)



    d.ellipse(

        [cx - ring_r, cy - ring_r, cx + ring_r, cy + ring_r],

        outline=(*CYAN2, 85),

        width=2,

    )



    d.ellipse(

        [cx - ring_r * 1.32, cy - ring_r * 1.32, cx + ring_r * 1.32, cy + ring_r * 1.32],

        outline=(*CYAN, 35),

        width=1,

    )



    # photon arcs

    for k, (start, extent, radius, alpha) in enumerate([

        (18, 74, 122, 120),

        (145, 52, 104, 90),

        (245, 82, 138, 110),

        (318, 38, 92, 70),

    ]):

        wobble = 4 * np.sin(phase * 2 + k)



        bbox = [

            cx - radius - wobble,

            cy - radius * 0.86,

            cx + radius + wobble,

            cy + radius * 0.86,

        ]



        d.arc(

            bbox,

            start=start + phase * 12,

            end=start + extent + phase * 12,

            fill=(*CYAN2, alpha),

            width=3 if k in (0, 2) else 2,

        )



    # central shadow / compact object

    for r, a in [(62, 28), (46, 42), (32, 90)]:

        d.ellipse(

            [cx - r, cy - r, cx + r, cy + r],

            outline=(*GREY, a),

            width=2,

        )



    d.ellipse(

        [cx - 25, cy - 25, cx + 25, cy + 25],

        fill=(0, 0, 0, 210),

    )



    d.ellipse(

        [cx - 29, cy - 29, cx + 29, cy + 29],

        outline=(*CYAN, 70),

        width=1,

    )



    # background source ghosts / lensed images

    sources = [

        (cx - 185, cy - 110, 0.7),

        (cx + 205, cy + 88, 0.9),

        (cx + 145, cy - 160, 0.5),

        (cx - 238, cy + 135, 0.6),

    ]



    for idx, (sx, sy, amp) in enumerate(sources):

        pulse = 0.55 + 0.45 * np.sin(phase * 2.4 + idx)



        # source marker

        d.ellipse(

            [sx - 3, sy - 3, sx + 3, sy + 3],

            fill=(*WHITE, int(80 * pulse)),

        )



        # lensed stretched image

        ang = np.arctan2(sy - cy, sx - cx) + np.pi / 2

        length = 36 * amp

        dx = np.cos(ang) * length

        dy = np.sin(ang) * length



        d.line(

            [sx - dx, sy - dy, sx + dx, sy + dy],

            fill=(*CYAN2, int(95 * pulse)),

            width=2,

        )



    # measurement ticks around ring

    for k in range(24):

        ang = phase * 0.25 + k * 2 * np.pi / 24

        r0 = ring_r + 22

        r1 = ring_r + 32 if k % 4 == 0 else ring_r + 27



        x0 = cx + np.cos(ang) * r0

        y0 = cy + np.sin(ang) * r0

        x1 = cx + np.cos(ang) * r1

        y1 = cy + np.sin(ang) * r1



        a = 55 if k % 4 else 120



        d.line([x0, y0, x1, y1], fill=(*CYAN, a), width=1)



    # label

    d.text(

        (32, 22),

        "GRAVITATIONAL LENSING FIELD",

        fill=(*CYAN2, 170),

    )



    d.text(

        (32, H - 34),

        "SPACETIME SHEAR MAP • EINSTEIN RING DETECTED • PHOTON PATH DISTORTION ACTIVE",

        fill=(*CYAN, 120),

    )



    # glow, but keep it mild

    glow = img.filter(ImageFilter.GaussianBlur(1.4))



    final = Image.new("RGBA", (W, H), BG)

    final.alpha_composite(glow)

    final.alpha_composite(img)



    return final





frames = [render_frame(i) for i in range(FRAMES)]



OUT = OUT_DIR / "gravity_lensing_overlay.gif"



frames[0].save(

    OUT,

    save_all=True,

    append_images=frames[1:],

    duration=int(1000 / FPS),

    loop=0,

    disposal=2,

    transparency=0,

)



print(f"Saved: {OUT}")


Saved: animations/gravity/gravity_lensing_overlay.gif


In [4]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/particles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 168

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(91)


def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]
    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT = load_font(24)
FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


# persistent particle tracks
particles = []
for _ in range(140):
    particles.append({
        "x0": rng.uniform(60, W - 300),
        "y0": rng.uniform(80, H - 70),
        "vx": rng.uniform(-0.4, 0.4),
        "vy": rng.uniform(-0.7, 0.7),
        "phase": rng.uniform(0, 2 * np.pi),
        "energy": rng.uniform(0.2, 1.0),
        "life": rng.integers(40, 120),
    })


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    # Header
    d.text((32, 10), "PARTICLE FLUX MONITOR", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 220, 34), fill=(*CYAN, 170), width=2)
    d.line((220, 34, 242, 48), fill=(*CYAN, 170), width=2)
    d.line((242, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Detector field grid
    field_x0, field_y0 = 48, 72
    field_x1, field_y1 = 625, H - 52

    d.rectangle([field_x0, field_y0, field_x1, field_y1], outline=(*CYAN, 70), width=1)

    for x in range(field_x0 + 48, field_x1, 48):
        d.line((x, field_y0, x, field_y1), fill=(*CYAN, 14), width=1)

    for y in range(field_y0 + 48, field_y1, 48):
        d.line((field_x0, y, field_x1, y), fill=(*CYAN, 14), width=1)

    # Particle hits
    flux_level = 0.55 + 0.35 * np.sin(phase * 1.4) ** 2
    spike = np.exp(-0.5 * ((i - 92) / 8) ** 2)
    flux_level += 0.75 * spike

    active_particles = int(35 + 85 * min(1.0, flux_level))

    for idx, p in enumerate(particles[:active_particles]):
        t = (i + idx * 7) % p["life"]
        a = 1.0 - t / p["life"]

        x = p["x0"] + p["vx"] * t + 8 * np.sin(phase + p["phase"])
        y = p["y0"] + p["vy"] * t + 6 * np.cos(phase * 1.2 + p["phase"])

        if not (field_x0 < x < field_x1 and field_y0 < y < field_y1):
            continue

        energy = p["energy"]
        color = CYAN if energy < 0.75 else YELLOW
        alpha = int((60 + 160 * energy) * a)

        r = 1 + 2.5 * energy

        d.ellipse(
            [x - r, y - r, x + r, y + r],
            fill=(*color, max(0, min(255, alpha))),
        )

        # streak for high energy particles
        if energy > 0.82:
            length = 18 + 30 * energy
            x2 = x - p["vx"] * length
            y2 = y - p["vy"] * length
            d.line(
                [x, y, x2, y2],
                fill=(*YELLOW, int(90 * a)),
                width=1,
            )

    # Rare high-energy flash
    if spike > 0.05:
        flash_alpha = int(150 * spike)
        d.rectangle([field_x0, field_y0, field_x1, field_y1], outline=(*RED, flash_alpha), width=3)
        d.text((field_x0 + 18, field_y0 + 16), "HIGH ENERGY EVENT", font=FONT_SMALL, fill=(*RED, flash_alpha + 40))

    # Histogram panel
    panel_x, panel_y = 675, 90
    panel_w, panel_h = 230, 360

    d.rectangle([panel_x, panel_y, panel_x + panel_w, panel_y + panel_h], outline=(*CYAN, 105), width=1)
    d.line((panel_x, panel_y, panel_x + 90, panel_y), fill=(*CYAN2, 190), width=3)

    d.text((panel_x + 16, panel_y + 16), "ENERGY FLUX", font=FONT_SMALL, fill=(*CYAN2, 190))

    # Live counters
    cpm = 380 + 220 * flux_level + 420 * spike
    dose = 0.17 + 0.09 * flux_level + 0.18 * spike
    threshold = "NOMINAL" if spike < 0.35 else "WARNING"
    threshold_color = GREEN if threshold == "NOMINAL" else RED

    metrics = [
        ("CPM", f"{cpm:06.1f}"),
        ("DOSE", f"{dose:.3f} mSv"),
        ("FLUX", f"{flux_level:.3f}"),
        ("STATE", threshold),
    ]

    for m, (k, v) in enumerate(metrics):
        y = panel_y + 55 + m * 38
        d.text((panel_x + 16, y), k, font=FONT_TINY, fill=(*CYAN, 155))
        col = threshold_color if k == "STATE" else CYAN2
        d.text((panel_x + 82, y), v, font=FONT_SMALL, fill=(*col, 220))

    # Histogram
    hist_x = panel_x + 18
    hist_y = panel_y + 235
    hist_w = panel_w - 36
    hist_h = 88

    d.rectangle([hist_x, hist_y, hist_x + hist_w, hist_y + hist_h], outline=(*CYAN, 60), width=1)

    bins = 18
    for b in range(bins):
        x = hist_x + 8 + b * 10
        base = 0.25 + 0.55 * np.exp(-((b - 6) / 5) ** 2)
        live = base + 0.18 * np.sin(phase * 3 + b * 0.7)
        live += spike * np.exp(-((b - 14) / 2.2) ** 2)
        live = np.clip(live, 0.05, 1.0)

        bar_h = int(live * (hist_h - 16))
        color = YELLOW if b > 13 else CYAN

        d.rectangle(
            [x, hist_y + hist_h - 8 - bar_h, x + 6, hist_y + hist_h - 8],
            fill=(*color, 160),
        )

    d.text((hist_x, hist_y - 18), "ENERGY SPECTRUM", font=FONT_TINY, fill=(*CYAN, 140))

    # Detector status ticks
    for t in range(18):
        x = field_x0 + 16 + t * 16
        a = int(40 + 100 * np.sin(phase * 5 + t) ** 2)
        d.line((x, field_y1 + 18, x + 8, field_y1 + 18), fill=(*CYAN, a), width=1)

    # Footer
    d.text(
        (32, H - 28),
        "PARTICLE IMPACT ARRAY • HIGH ENERGY COUNTER • RADIATION THRESHOLD MONITOR",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    # Corners
    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.8))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "particle_flux_monitor.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/particles/particle_flux_monitor.gif


In [5]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/radar")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 720, 720
FPS = 24
FRAMES = 168

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(123)

def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]
    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()

FONT = load_font(20)
FONT_SMALL = load_font(13)
FONT_TINY = load_font(11)

CONTACTS = [
    {"r": 0.32, "a": 0.35, "id": "C-17", "type": "LOCK", "color": GREEN},
    {"r": 0.52, "a": 1.42, "id": "X-04", "type": "UNKNOWN", "color": YELLOW},
    {"r": 0.71, "a": 2.66, "id": "D-91", "type": "DRIFT", "color": CYAN2},
    {"r": 0.46, "a": 4.20, "id": "R-22", "type": "RETURN", "color": CYAN},
    {"r": 0.82, "a": 5.18, "id": "A-08", "type": "FAST", "color": RED},
]

def polar_to_xy(cx, cy, radius, angle):
    return cx + np.cos(angle) * radius, cy + np.sin(angle) * radius

def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    cx = cy = W // 2
    max_r = 285

    d.text((28, 16), "SUBSPACE RADAR", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((24, 40, 170, 40), fill=(*CYAN, 170), width=2)
    d.line((170, 40, 192, 54), fill=(*CYAN, 170), width=2)
    d.line((192, 54, W - 30, 54), fill=(*CYAN, 70), width=1)

    # Radar rings
    for r in [70, 130, 190, 250, max_r]:
        d.ellipse([cx-r, cy-r, cx+r, cy+r], outline=(*CYAN, 45), width=1)

    # radial grid
    for k in range(24):
        a = k * 2 * np.pi / 24
        x, y = polar_to_xy(cx, cy, max_r, a)
        alpha = 55 if k % 3 == 0 else 24
        d.line((cx, cy, x, y), fill=(*CYAN, alpha), width=1)

    # interference sector
    sector_start = 215 + 10 * np.sin(phase * 0.7)
    d.pieslice(
        [cx-max_r, cy-max_r, cx+max_r, cy+max_r],
        start=sector_start,
        end=sector_start + 28,
        fill=(*RED, 16),
    )

    # rotating sweep
    sweep_angle = phase * 360 / (2 * np.pi)

    sweep = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    sd = ImageDraw.Draw(sweep)

    for k in range(42):
        alpha = int(5 + (42 - k) * 2.2)
        sd.pieslice(
            [cx-max_r, cy-max_r, cx+max_r, cy+max_r],
            start=sweep_angle - k * 1.6,
            end=sweep_angle - k * 1.6 + 1.2,
            fill=(*CYAN, alpha),
        )

    sweep = sweep.filter(ImageFilter.GaussianBlur(7))
    img.alpha_composite(sweep)

    # echo trails / contacts
    for idx, c in enumerate(CONTACTS):
        drift = 0.035 * np.sin(phase * (0.7 + idx * 0.12) + idx)
        angle = c["a"] + drift
        radius = max_r * c["r"] * (1 + 0.025 * np.sin(phase * 1.3 + idx))
        x, y = polar_to_xy(cx, cy, radius, angle)

        contact_phase = (sweep_angle / 360 * 2 * np.pi - angle) % (2 * np.pi)
        echo = np.exp(-0.5 * (min(contact_phase, 2*np.pi-contact_phase) / 0.28) ** 2)
        pulse = 0.55 + 0.45 * np.sin(phase * 5 + idx)

        color = c["color"]
        alpha = int(90 + 130 * max(echo, pulse * 0.45))

        # persistent ghost echo
        for g in range(1, 5):
            ga = angle - g * 0.07
            gx, gy = polar_to_xy(cx, cy, radius, ga)
            gr = 6 + g * 3
            d.ellipse(
                [gx-gr, gy-gr, gx+gr, gy+gr],
                outline=(*color, max(20, alpha - g * 35)),
                width=1,
            )

        # main contact
        r = 5 + 4 * echo
        d.ellipse([x-r, y-r, x+r, y+r], fill=(*color, alpha))

        # target bracket
        b = 18 + 5 * pulse
        d.line((x-b, y-b, x-b+9, y-b), fill=(*color, alpha), width=1)
        d.line((x-b, y-b, x-b, y-b+9), fill=(*color, alpha), width=1)
        d.line((x+b, y-b, x+b-9, y-b), fill=(*color, alpha), width=1)
        d.line((x+b, y-b, x+b, y-b+9), fill=(*color, alpha), width=1)
        d.line((x-b, y+b, x-b+9, y+b), fill=(*color, alpha), width=1)
        d.line((x-b, y+b, x-b, y+b-9), fill=(*color, alpha), width=1)
        d.line((x+b, y+b, x+b-9, y+b), fill=(*color, alpha), width=1)
        d.line((x+b, y+b, x+b, y+b-9), fill=(*color, alpha), width=1)

        # labels
        if echo > 0.25 or idx < 3:
            d.text((x + 22, y - 12), c["id"], font=FONT_TINY, fill=(*color, 190))
            d.text((x + 22, y + 2), c["type"], font=FONT_TINY, fill=(*CYAN2, 145))

        # velocity vector
        vx = -np.sin(angle) * 26
        vy = np.cos(angle) * 26
        d.line((x, y, x + vx, y + vy), fill=(*GREEN, 120), width=1)

    # center station
    for r, a in [(24, 45), (14, 90), (5, 220)]:
        d.ellipse([cx-r, cy-r, cx+r, cy+r], outline=(*WHITE, a), width=2)
    d.ellipse([cx-4, cy-4, cx+4, cy+4], fill=(*WHITE, 240))

    # bottom telemetry
    d.text(
        (30, H - 52),
        f"CONTACTS {len(CONTACTS):02d}  SWEEP {int(sweep_angle)%360:03d}°  INTERFERENCE {18 + 6*np.sin(phase*2):04.1f}%",
        font=FONT_TINY,
        fill=(*CYAN, 150),
    )

    d.text(
        (30, H - 30),
        "ECHO PERSISTENCE ACTIVE • UNKNOWN RETURN FILTER • VECTOR TRACKING ONLINE",
        font=FONT_TINY,
        fill=(*CYAN, 110),
    )

    # side packet ticks
    for k in range(18):
        a = int(35 + 90 * np.sin(phase * 4 + k) ** 2)
        d.line((W - 86 + k * 4, 84, W - 84 + k * 4, 84), fill=(*CYAN, a), width=1)

    # HUD corners
    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.8))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")

frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "subspace_radar.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/radar/subspace_radar.gif


In [8]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/stellar")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 168

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
ORANGE = (255, 170, 70)
YELLOW = (255, 230, 110)
RED = (255, 80, 80)
WHITE = (245, 250, 255)

rng = np.random.default_rng(210)


def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]
    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT = load_font(24)
FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    cx, cy = 340, 270
    r_star = 165

    d.text((32, 10), "STELLAR CORE MONITOR", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 215, 34), fill=(*CYAN, 170), width=2)
    d.line((215, 34, 238, 48), fill=(*CYAN, 170), width=2)
    d.line((238, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Reference rings
    for rr, a in [(r_star, 90), (118, 45), (72, 55), (34, 70)]:
        d.ellipse([cx-rr, cy-rr, cx+rr, cy+rr], outline=(*CYAN, a), width=1)

    # Convection cells
    for k in range(72):
        ang = k * 2 * np.pi / 72 + 0.18 * np.sin(phase + k)
        rr = rng.uniform(42, r_star - 10)
        cell_r = rng.uniform(8, 20)

        x = cx + np.cos(ang + phase * 0.05) * rr
        y = cy + np.sin(ang + phase * 0.05) * rr

        if (x-cx)**2 + (y-cy)**2 > r_star**2:
            continue

        pulse = 0.45 + 0.55 * np.sin(phase * (1.2 + k % 5 * 0.13) + k)
        alpha = int(55 + 105 * pulse)

        color = YELLOW if rr < 85 else ORANGE

        d.ellipse(
            [x-cell_r, y-cell_r, x+cell_r, y+cell_r],
            fill=(*color, alpha),
        )

    core_pulse = 0.65 + 0.35 * np.sin(phase * 2.3)
    
    # Core glow
    for rr, a in [(56, 40), (38, 90), (18, 210)]:

        col = WHITE if rr < 20 else YELLOW

        d.ellipse(
            [cx-rr, cy-rr, cx+rr, cy+rr],
            fill=(*col, int(a * core_pulse)),
        )


    # Radial pressure waves
    for k in range(4):
        wr = 42 + ((i * 3 + k * 48) % 170)
        alpha = max(0, 95 - int(wr * 0.38))
        d.ellipse(
            [cx-wr, cy-wr, cx+wr, cy+wr],
            outline=(*CYAN2, alpha),
            width=2,
        )

    # Instability flash
    instability = np.exp(-0.5 * ((i - 104) / 8) ** 2)
    if instability > 0.04:
        a = int(170 * instability)
        d.ellipse([cx-r_star-10, cy-r_star-10, cx+r_star+10, cy+r_star+10], outline=(*RED, a), width=4)
        d.text((cx - 92, cy - r_star - 34), "CORE INSTABILITY", font=FONT_SMALL, fill=(*RED, a + 40))

    # Magnetic / plasma arcs
    for k in range(6):
        start = phase * 30 + k * 60
        d.arc(
            [cx-r_star-22, cy-r_star*0.62-22, cx+r_star+22, cy+r_star*0.62+22],
            start=start,
            end=start + 28,
            fill=(*CYAN2, 80),
            width=2,
        )

    # Right telemetry panel
    px, py = 660, 92
    pw, ph = 250, 340

    d.rectangle([px, py, px+pw, py+ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px+95, py), fill=(*CYAN2, 190), width=3)

    d.text((px + 16, py + 16), "CORE STATE", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("TEMP", f"{15.4 + 0.7*np.sin(phase):04.1f} MK"),
        ("PRESS", f"{2.31 + 0.08*np.cos(phase*1.4):.2f}e16"),
        ("FUSION", f"{92 + 5*np.sin(phase*1.8):04.1f}%"),
        ("ν FLUX", f"{6.4 + 0.6*np.sin(phase*2.1):.2f}e10"),
        ("CONV", f"{74 + 12*np.sin(phase*1.2):04.1f}%"),
        ("STAB", "WARN" if instability > 0.25 else "NOMINAL"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 42
        col = RED if k == "STAB" and v == "WARN" else CYAN2

        d.text((px + 18, y), k, font=FONT_TINY, fill=(*CYAN, 155))
        d.text((px + 100, y), v, font=FONT_SMALL, fill=(*col, 225))

        for t in range(6):
            a = int(35 + 90 * np.sin(phase * 4 + m + t) ** 2)
            tx = px + 205 + t * 6
            d.line((tx, y + 8, tx + 3, y + 8), fill=(*CYAN, a), width=1)

    d.text(
        (32, H - 30),
        "CONVECTION MODEL ACTIVE • NEUTRINO FLUX STREAM • CORE PRESSURE WAVES DETECTED",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.8))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "stellar_core_monitor.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/stellar/stellar_core_monitor.gif


In [10]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/black_holes")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
ORANGE = (255, 160, 70)
YELLOW = (255, 225, 95)
RED = (255, 80, 80)
GREY = (120, 135, 145)
WHITE = (245, 250, 255)


def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]
    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    cx, cy = 365, 270

    d.text((32, 10), "EVENT HORIZON SCAN", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 205, 34), fill=(*CYAN, 170), width=2)
    d.line((205, 34, 228, 48), fill=(*CYAN, 170), width=2)
    d.line((228, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # warped reference rings
    for r, a in [(75, 24), (125, 30), (180, 28), (235, 22)]:
        wobble = 7 * np.sin(phase * 1.2 + r)
        d.ellipse(
            [cx - r, cy - r * 0.58 + wobble, cx + r, cy + r * 0.58 + wobble],
            outline=(*GREY, a),
            width=1,
        )

    # accretion disk rear glow
    disk = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    dd = ImageDraw.Draw(disk)

    for k in range(34):
        angle = phase * 28 + k * 10
        alpha = int(35 + 80 * np.sin(phase * 2.1 + k) ** 2)
        col = ORANGE if k % 3 else YELLOW

        dd.arc(
            [cx - 245, cy - 88, cx + 245, cy + 88],
            start=angle,
            end=angle + 36,
            fill=(*col, alpha),
            width=3,
        )

    disk = disk.filter(ImageFilter.GaussianBlur(3.2))
    img.alpha_composite(disk)

    # photon ring
    for r, a, w in [(82, 95, 2), (96, 55, 1), (112, 30, 1)]:
        d.ellipse(
            [cx - r, cy - r, cx + r, cy + r],
            outline=(*CYAN2, a),
            width=w,
        )

    # black shadow
    d.ellipse([cx - 74, cy - 74, cx + 74, cy + 74], fill=(0, 0, 0, 245))
    d.ellipse([cx - 78, cy - 78, cx + 78, cy + 78], outline=(*CYAN, 65), width=1)

    # front accretion arcs
    for k in range(10):
        start = phase * 45 + k * 36
        radius = 180 + 14 * np.sin(phase * 1.7 + k)

        d.arc(
            [cx - radius, cy - radius * 0.34, cx + radius, cy + radius * 0.34],
            start=start,
            end=start + 24,
            fill=(*ORANGE, 110 if k % 2 else 75),
            width=2,
        )

    # lensing arcs
    for k, rad in enumerate([128, 156, 205]):
        start = 25 + k * 85 + phase * 11
        d.arc(
            [cx - rad, cy - rad * 0.82, cx + rad, cy + rad * 0.82],
            start=start,
            end=start + 62,
            fill=(*CYAN2, 70 - k * 12),
            width=2,
        )


    # unstable signal ticks
    for k in range(34):
        x = 85 + k * 15
        y = 475 + 8 * np.sin(phase * 4 + k)
        a = int(40 + 100 * np.sin(phase * 5 + k) ** 2)
        d.line((x, y, x + 7, y), fill=(*CYAN, a), width=1)

    # right telemetry panel
    px, py = 675, 92
    pw, ph = 245, 335

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 100), width=1)
    d.line((px, py, px + 95, py), fill=(*CYAN2, 190), width=3)

    d.text((px + 16, py + 16), "HORIZON FIT", font=FONT_SMALL, fill=(*CYAN2, 190))

    instability = np.exp(-0.5 * ((i - 103) / 9) ** 2)

    metrics = [
        ("RING", f"{42.1 + 0.6*np.sin(phase):04.1f} μas"),
        ("SPIN", f"{0.82 + 0.03*np.cos(phase):.2f} a"),
        ("SHEAR", f"{0.31 + 0.04*np.sin(phase*1.3):.3f}"),
        ("SNR", f"{37 + 8*np.sin(phase*2):04.1f} dB"),
        ("LOSS", f"{2.4 + 14*instability:04.1f}%"),
        ("STATE", "UNSTABLE" if instability > 0.35 else "LOCKED"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 42
        col = RED if k == "STATE" and v == "UNSTABLE" else CYAN2

        d.text((px + 18, y), k, font=FONT_TINY, fill=(*CYAN, 155))
        d.text((px + 95, y), v, font=FONT_SMALL, fill=(*col, 225))

    if instability > 0.05:
        a = int(150 * instability)
        d.rectangle([px, py, px + pw, py + ph], outline=(*RED, a), width=3)
        d.text((px + 20, py + ph - 32), "SIGNAL SHEAR EVENT", font=FONT_TINY, fill=(*RED, a + 40))

    d.text(
        (32, H - 28),
        "PHOTON RING RECONSTRUCTION • RELATIVISTIC SHEAR MAP • ACCRETION FLOW LOCK",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.7))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "event_horizon_scan.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/black_holes/event_horizon_scan.gif


In [3]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/archive")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)
CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)

rng = np.random.default_rng(404)


def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]

    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass

    return ImageFont.load_default()


FONT = load_font(22)
FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)

GLYPHS = list("◈◇◆△▽◁▷◎◉⊙⊕⊗⌬⌁⌘⌑⌖⌗⟡⟐⟁⧫⧉⧗")


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


# ----------------------------
# Archive matrix layout
# ----------------------------
cols, rows = 9, 5

frame_x0, frame_y0 = 44, 68
frame_x1, frame_y1 = 690, 420

pad_x = 18
pad_y = 18
gap = 8

grid_x0 = frame_x0 + pad_x
grid_y0 = frame_y0 + pad_y

grid_w = (frame_x1 - frame_x0) - pad_x * 2
grid_h = (frame_y1 - frame_y0) - pad_y * 2

cell_w = (grid_w - gap * (cols - 1)) / cols
cell_h = (grid_h - gap * (rows - 1)) / rows

sectors = []

for r in range(rows):
    for c in range(cols):
        sectors.append({
            "x": grid_x0 + c * (cell_w + gap),
            "y": grid_y0 + r * (cell_h + gap),
            "delay": 18 + (r * cols + c) * 2.2 + rng.uniform(-4, 4),
            "glyph": rng.choice(GLYPHS),
            "damage": rng.random() < 0.22,
        })


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES
    restore = smoothstep((i - 20) / 115)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    # Header
    d.text(
        (32, 10),
        "DEEP ARCHIVE RECOVERY",
        font=FONT_SMALL,
        fill=(*CYAN2, 190),
    )

    d.line((28, 34, 225, 34), fill=(*CYAN, 170), width=2)
    d.line((225, 34, 248, 48), fill=(*CYAN, 170), width=2)
    d.line((248, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Archive matrix frame
    d.rectangle(
        [frame_x0, frame_y0, frame_x1, frame_y1],
        outline=(*CYAN, 80),
        width=1,
    )

    restored_count = 0
    damaged_count = 0

    for idx, s in enumerate(sectors):
        a = smoothstep((i - s["delay"]) / 14)

        if a <= 0:
            continue

        x, y = s["x"], s["y"]
        damage = s["damage"] and i < 142

        if a > 0.9 and not damage:
            restored_count += 1

        if damage:
            damaged_count += 1

        pulse = 0.55 + 0.45 * np.sin(phase * 4 + idx)
        outline = RED if damage else CYAN
        fill_alpha = int((18 + 32 * pulse) * a)

        # Sector block
        d.rectangle(
            [x, y, x + cell_w, y + cell_h],
            outline=(*outline, int(95 * a)),
            fill=(*CYAN, fill_alpha),
            width=1,
        )

        if damage:
            d.line(
                [x + 6, y + 8, x + cell_w - 6, y + cell_h - 8],
                fill=(*RED, int(120 * a)),
                width=1,
            )
            d.line(
                [x + 6, y + cell_h - 8, x + cell_w - 6, y + 8],
                fill=(*RED, int(90 * a)),
                width=1,
            )

        glyph = s["glyph"]

        if damage and int(i / 4) % 2 == 0:
            glyph = rng.choice(GLYPHS)

        d.text(
            (x + cell_w * 0.32, y + cell_h * 0.22),
            glyph,
            font=FONT,
            fill=(*CYAN2, int(220 * a)),
        )

        # Micro data line
        line_y = y + cell_h - 10

        for k in range(5):
            aa = int((35 + 80 * np.sin(phase * 5 + idx + k) ** 2) * a)
            lx = x + 8 + k * 10

            d.line(
                [lx, line_y, lx + 6, line_y],
                fill=(*CYAN, aa),
                width=1,
            )

    # Reconstruction geometry overlay
    if restore > 0.25:
        geom_a = int(90 * restore)
        center = (365, 245)

        for rr in [70, 112, 156]:
            d.ellipse(
                [
                    center[0] - rr,
                    center[1] - rr * 0.55,
                    center[0] + rr,
                    center[1] + rr * 0.55,
                ],
                outline=(*CYAN2, geom_a // 2),
                width=1,
            )

        for k in range(12):
            ang = phase * 0.25 + k * 2 * np.pi / 12
            x1 = center[0] + np.cos(ang) * 170
            y1 = center[1] + np.sin(ang) * 92

            d.line(
                [center[0], center[1], x1, y1],
                fill=(*CYAN, geom_a // 3),
                width=1,
            )

    # Right recovery panel
    px, py = 725, 92
    pw, ph = 195, 300

    d.rectangle(
        [px, py, px + pw, py + ph],
        outline=(*CYAN, 105),
        width=1,
    )
    d.line((px, py, px + 88, py), fill=(*CYAN2, 190), width=3)

    d.text(
        (px + 16, py + 16),
        "ARCHIVE INDEX",
        font=FONT_SMALL,
        fill=(*CYAN2, 190),
    )

    progress = min(100, int(restore * 100))

    metrics = [
        ("RESTORE", f"{progress:03d}%"),
        ("SECTORS", f"{restored_count:02d}/{len(sectors):02d}"),
        ("DAMAGE", f"{damaged_count:02d}"),
        ("INDEX", "LOCK" if i > 145 else "SCAN"),
        ("LANG", "UNKNOWN"),
        ("STATE", "RESTORED" if i > 150 else "RECOVER"),
    ]

    for m, (k, v) in enumerate(metrics):
        yy = py + 55 + m * 36

        col = GREEN if v in ("LOCK", "RESTORED") else CYAN2

        if k == "DAMAGE" and damaged_count > 0:
            col = YELLOW

        d.text(
            (px + 14, yy),
            k,
            font=FONT_TINY,
            fill=(*CYAN, 150),
        )

        d.text(
            (px + 86, yy),
            v,
            font=FONT_SMALL,
            fill=(*col, 220),
        )

    # Progress bar
    bx, by = px + 16, py + ph - 32
    bw, bh = pw - 32, 12

    d.rectangle(
        [bx, by, bx + bw, by + bh],
        outline=(*CYAN, 95),
        width=1,
    )

    d.rectangle(
        [
            bx + 2,
            by + 2,
            bx + 2 + int((bw - 4) * restore),
            by + bh - 2,
        ],
        fill=(*GREEN, 170),
    )

    # Occasional corruption glitch
    if 75 < i < 92 or 122 < i < 132:
        for _ in range(12):
            gy = int(rng.integers(frame_y0 + 2, frame_y1 - 2))
            gx = int(rng.integers(frame_x0 + 2, frame_x1 - 120))
            gw = int(rng.integers(50, 220))

            d.rectangle(
                [gx, gy, min(frame_x1 - 2, gx + gw), gy + 4],
                fill=(*RED, int(rng.integers(30, 90))),
            )

    # Footer
    d.text(
        (32, H - 30),
        "FRAGMENTED SYMBOL INDEX • DAMAGED SECTORS ISOLATED • ARCHIVE GEOMETRY RESTORED",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    # HUD corners
    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.8))

    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "deep_archive_loader.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/archive/deep_archive_loader.gif


In [4]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/bio")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 168

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]
    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()

FONT = load_font(24)
FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)

CHANNELS = [
    ("CARDIO", GREEN),
    ("NEURAL", CYAN2),
    ("OXYGEN", CYAN),
    ("STRESS", YELLOW),
    ("SYNTH", RED),
]

def waveform(x, phase, idx):
    base = (
        0.45 * np.sin(x * 0.035 + phase * (1.0 + idx * 0.12))
        + 0.18 * np.sin(x * 0.12 - phase * 1.7)
    )

    if idx == 0:
        spike_pos = (phase * 45) % 260
        spike = 1.2 * np.exp(-0.5 * ((x - spike_pos) / 3.2) ** 2)
        return base * 0.35 + spike

    if idx == 1:
        return base + 0.18 * np.sin(x * 0.42 + phase * 4)

    if idx == 3:
        anomaly = 0.9 * np.exp(-0.5 * ((x - 180) / 13) ** 2) * np.sin(phase * 1.4) ** 2
        return base * 0.55 + anomaly

    return base * 0.65

def draw_trace(d, x0, y0, w, amp, phase, color, idx):
    pts = []

    for x in range(w):
        yy = waveform(x, phase, idx)
        pts.append((x0 + x, y0 - yy * amp))

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*color, 65), width=4)

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*color, 220), width=1)

def draw_body_silhouette(d, cx, cy, phase):
    pulse = 0.55 + 0.45 * np.sin(phase * 2.2)

    # head
    d.ellipse(
        [cx - 34, cy - 155, cx + 34, cy - 87],
        outline=(*CYAN2, 135),
        width=2,
    )

    # torso
    d.rounded_rectangle(
        [cx - 58, cy - 80, cx + 58, cy + 95],
        radius=42,
        outline=(*CYAN, 115),
        width=2,
    )

    # spine / central scan
    d.line([cx, cy - 72, cx, cy + 82], fill=(*CYAN2, int(120 + 70 * pulse)), width=1)

    # shoulders/arms
    d.arc([cx - 118, cy - 68, cx - 32, cy + 88], 250, 80, fill=(*CYAN, 80), width=2)
    d.arc([cx + 32, cy - 68, cx + 118, cy + 88], 100, 290, fill=(*CYAN, 80), width=2)

    # scan rings
    for r, a in [(86, 32), (124, 22), (164, 16)]:
        rr = r + 4 * np.sin(phase * 2.5 + r)
        d.ellipse(
            [cx - rr, cy - rr * 1.18, cx + rr, cy + rr * 1.18],
            outline=(*CYAN, a),
            width=1,
        )

    # body target points
    points = [
        (cx, cy - 122),
        (cx, cy - 18),
        (cx - 34, cy + 42),
        (cx + 34, cy + 42),
    ]

    for k, (x, y) in enumerate(points):
        a = int(110 + 90 * np.sin(phase * 4 + k) ** 2)
        d.ellipse([x - 5, y - 5, x + 5, y + 5], fill=(*GREEN, a))

def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    # Header
    d.text((32, 10), "BIOSCANNER HUD", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 180, 34), fill=(*CYAN, 170), width=2)
    d.line((180, 34, 202, 48), fill=(*CYAN, 170), width=2)
    d.line((202, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Left scan target
    draw_body_silhouette(d, 250, 280, phase)

    # Vertical sweep over body
    scan_y = 95 + int((i * 4) % 350)
    for dy in range(-5, 6):
        a = max(0, 42 - abs(dy) * 7)
        d.line((95, scan_y + dy, 405, scan_y + dy), fill=(*CYAN2, a), width=1)

    # Trace panel
    trace_x0 = 470
    trace_y0 = 92
    trace_w = 410
    row_h = 70

    d.rectangle([445, 74, 910, 435], outline=(*CYAN, 90), width=1)
    d.line((445, 74, 545, 74), fill=(*CYAN2, 190), width=3)

    for idx, (name, color) in enumerate(CHANNELS):
        y = trace_y0 + idx * row_h

        d.text((465, y - 22), name, font=FONT_TINY, fill=(*color, 175))
        d.line((trace_x0, y, trace_x0 + trace_w, y), fill=(*CYAN, 30), width=1)

        draw_trace(
            d,
            trace_x0,
            y,
            trace_w,
            18 if idx != 0 else 24,
            phase,
            color,
            idx,
        )

        # right status dot
        status_col = GREEN if idx < 3 else (YELLOW if idx == 3 else RED)
        a = int(120 + 90 * np.sin(phase * 4 + idx) ** 2)
        d.ellipse([884, y - 7, 898, y + 7], fill=(*status_col, a))

    # Bottom metrics
    metrics = [
        ("PULSE", f"{72 + 5*np.sin(phase*1.7):03.0f} BPM"),
        ("O₂", f"{98.1 + 0.4*np.sin(phase):04.1f}%"),
        ("NEURAL", f"{82 + 8*np.sin(phase*2.1):04.1f}%"),
        ("STRESS", f"{31 + 18*np.sin(phase*1.2)**2:04.1f}%"),
        ("SYNTH LOAD", f"{44 + 22*np.sin(phase*0.8)**2:04.1f}%"),
    ]

    mx, my = 80, 455
    for idx, (k, v) in enumerate(metrics):
        x = mx + idx * 170
        d.text((x, my), k, font=FONT_TINY, fill=(*CYAN, 145))
        d.text((x, my + 20), v, font=FONT_SMALL, fill=(*CYAN2, 220))

    # Anomaly marker
    anomaly = np.exp(-0.5 * ((i - 104) / 10) ** 2)
    if anomaly > 0.06:
        a = int(180 * anomaly)
        d.rectangle([445, 74, 910, 435], outline=(*RED, a), width=3)
        d.text((466, 412), "ANOMALOUS STRESS SPIKE", font=FONT_TINY, fill=(*RED, min(255, a + 40)))

    # Footer
    d.text(
        (32, H - 28),
        "BIOELECTRIC STREAM • NEURAL RHYTHM LOCK • SYNTHETIC LOAD MONITOR ACTIVE",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    # HUD corners
    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.7))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")

frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "bioscanner_hud.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/bio/bioscanner_hud.gif


In [5]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/decoder")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(777)


def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]

    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass

    return ImageFont.load_default()


FONT = load_font(24)
FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)

GLYPHS = list("01ZXΔΛΣΩΨΦΞ∴∵∿≋⋈⊕⊗⌬⌁")


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


cols, rows = 18, 9
cell_w, cell_h = 30, 30
grid_x0, grid_y0 = 54, 84

matrix = rng.choice(GLYPHS, size=(rows, cols))


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES
    lock = smoothstep((i - 100) / 45)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((32, 10), "QUANTUM DECODER", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 185, 34), fill=(*CYAN, 170), width=2)
    d.line((185, 34, 208, 48), fill=(*CYAN, 170), width=2)
    d.line((208, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Matrix frame
    d.rectangle(
        [42, 70, 625, 380],
        outline=(*CYAN, 90),
        width=1,
    )

    # Matrix decode wave
    for r in range(rows):
        for c in range(cols):
            x = grid_x0 + c * cell_w
            y = grid_y0 + r * cell_h

            wave = smoothstep((i - 18 - (r + c) * 3.2) / 18)

            if wave <= 0:
                continue

            flicker = 0.55 + 0.45 * np.sin(phase * 6 + r * 1.2 + c)
            alpha = int((70 + 150 * flicker) * wave)

            if lock > 0.65 and (r in [2, 4, 6]) and (4 <= c <= 13):
                char = "█" if (c + r + i // 12) % 3 == 0 else " "
                col = GREEN
            else:
                if i % 5 == 0 and rng.random() < 0.02:
                    matrix[r, c] = rng.choice(GLYPHS)
                char = matrix[r, c]
                col = CYAN2 if flicker > 0.72 else CYAN

            d.text(
                (x, y),
                char,
                font=FONT_SMALL,
                fill=(*col, alpha),
            )

    # Interference bands
    for k in range(9):
        y = 410 + k * 8
        amp = 35 + 20 * np.sin(phase * 1.4 + k)
        pts = []

        for x in range(60, 620, 4):
            yy = y + np.sin(x * 0.035 + phase * 2 + k) * amp * 0.15
            pts.append((x, yy))

        for p1, p2 in zip(pts[:-1], pts[1:]):
            d.line([p1, p2], fill=(*CYAN, 55), width=1)

    # Collapsing pattern geometry
    cx, cy = 340, 225

    if lock > 0.1:
        a = int(120 * lock)

        for rr in [48, 92, 138]:
            d.ellipse(
                [cx - rr, cy - rr * 0.55, cx + rr, cy + rr * 0.55],
                outline=(*GREEN, a // 2),
                width=1,
            )

        for k in range(10):
            ang = phase * 0.35 + k * 2 * np.pi / 10
            x1 = cx + np.cos(ang) * 165
            y1 = cy + np.sin(ang) * 88

            d.line(
                [cx, cy, x1, y1],
                fill=(*GREEN, a // 3),
                width=1,
            )

    # Right panel
    px, py = 675, 92
    pw, ph = 245, 320

    d.rectangle(
        [px, py, px + pw, py + ph],
        outline=(*CYAN, 100),
        width=1,
    )
    d.line((px, py, px + 95, py), fill=(*CYAN2, 190), width=3)

    d.text((px + 16, py + 16), "PATTERN SOLVER", font=FONT_SMALL, fill=(*CYAN2, 190))

    coherence = 37 + 61 * lock + 3 * np.sin(phase * 2)
    entropy = 91 - 68 * lock + 4 * np.cos(phase * 1.3)
    phase_lock = 100 * lock

    metrics = [
        ("COHER", f"{coherence:04.1f}%"),
        ("ENTROPY", f"{entropy:04.1f}%"),
        ("PHASE", f"{phase_lock:04.1f}%"),
        ("LANG", "UNKNOWN"),
        ("STATE", "LOCK" if lock > 0.85 else "SOLVE"),
    ]

    for m, (k, v) in enumerate(metrics):
        yy = py + 58 + m * 42
        col = GREEN if v == "LOCK" else CYAN2

        d.text((px + 18, yy), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 100, yy), v, font=FONT_SMALL, fill=(*col, 220))

    # Progress bar
    bx, by = px + 18, py + ph - 34
    bw, bh = pw - 36, 12

    d.rectangle([bx, by, bx + bw, by + bh], outline=(*CYAN, 90), width=1)
    d.rectangle([bx + 2, by + 2, bx + 2 + int((bw - 4) * lock), by + bh - 2], fill=(*GREEN, 170))

    # Occasional decoherence glitch
    if 72 < i < 84 or 132 < i < 140:
        for _ in range(10):
            gx = int(rng.integers(48, 620))
            gy = int(rng.integers(76, 370))
            gw = int(rng.integers(40, 170))
            d.rectangle([gx, gy, gx + gw, gy + 3], fill=(*RED, int(rng.integers(25, 85))))

    d.text(
        (32, H - 30),
        "WAVEFORM COLLAPSE • SYMBOL MATRIX SEARCH • UNKNOWN LANGUAGE PATTERN LOCK",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.7))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "quantum_decoder.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/decoder/quantum_decoder.gif


In [7]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/tracking")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 168

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(1337)


def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]

    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass

    return ImageFont.load_default()


FONT = load_font(22)
FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


TARGETS = [
    {"id": "TGT-01", "type": "FAST", "x": 140, "y": 120, "vx": 2.9, "vy": 1.1, "color": RED},
    {"id": "TGT-02", "type": "DRIFT", "x": 780, "y": 140, "vx": -1.8, "vy": 1.5, "color": YELLOW},
    {"id": "TGT-03", "type": "LOCK", "x": 260, "y": 410, "vx": 1.6, "vy": -1.7, "color": GREEN},
    {"id": "TGT-04", "type": "UNK", "x": 720, "y": 410, "vx": -2.2, "vy": -1.2, "color": CYAN2},
    {"id": "TGT-05", "type": "GHOST", "x": 470, "y": 250, "vx": 0.9, "vy": 0.5, "color": CYAN},
]


def wrap_position(x, y):
    margin_left = 290
    margin_right = 70
    margin_y = 70

    x = ((x - margin_left) % (W - margin_left - margin_right)) + margin_left
    y = ((y - margin_y) % (H - 2 * margin_y)) + margin_y

    return x, y

def target_state(t, frame_idx):
    x = t["x"] + t["vx"] * frame_idx
    y = t["y"] + t["vy"] * frame_idx

    x += 10 * np.sin(frame_idx * 0.04 + t["x"])
    y += 8 * np.cos(frame_idx * 0.05 + t["y"])

    return wrap_position(x, y)


def draw_brackets(d, x, y, size, color, alpha=220):
    c = size * 0.38

    # TL
    d.line((x-size, y-size, x-size+c, y-size), fill=(*color, alpha), width=2)
    d.line((x-size, y-size, x-size, y-size+c), fill=(*color, alpha), width=2)

    # TR
    d.line((x+size, y-size, x+size-c, y-size), fill=(*color, alpha), width=2)
    d.line((x+size, y-size, x+size, y-size+c), fill=(*color, alpha), width=2)

    # BL
    d.line((x-size, y+size, x-size+c, y+size), fill=(*color, alpha), width=2)
    d.line((x-size, y+size, x-size, y+size-c), fill=(*color, alpha), width=2)

    # BR
    d.line((x+size, y+size, x+size-c, y+size), fill=(*color, alpha), width=2)
    d.line((x+size, y+size, x+size, y+size-c), fill=(*color, alpha), width=2)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    # Header
    d.text((32, 10), "MULTI TARGET TRACKER", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 225, 34), fill=(*CYAN, 170), width=2)
    d.line((225, 34, 248, 48), fill=(*CYAN, 170), width=2)
    d.line((248, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Grid
    for x in range(80, W, 80):
        d.line((x, 72, x, H - 54), fill=(*CYAN, 14), width=1)

    for y in range(90, H - 50, 60):
        d.line((42, y, W - 42, y), fill=(*CYAN, 14), width=1)

    # Current target positions
    states = []

    for t in TARGETS:
        x, y = target_state(t, i)
        states.append((t, x, y))

    # Conflict line between closest pair
    closest = None
    closest_dist = 999999

    for a in range(len(states)):
        for b in range(a + 1, len(states)):
            _, x1, y1 = states[a]
            _, x2, y2 = states[b]

            dist = np.hypot(x2 - x1, y2 - y1)

            if dist < closest_dist:
                closest_dist = dist
                closest = (states[a], states[b])

    if closest and closest_dist < 260:
        (_, x1, y1), (_, x2, y2) = closest
        warn_alpha = int(90 + 90 * np.sin(phase * 5) ** 2)

        d.line((x1, y1, x2, y2), fill=(*RED, warn_alpha), width=1)
        d.text(
            ((x1 + x2) / 2 + 12, (y1 + y2) / 2 - 8),
            "CROSSING VECTOR",
            font=FONT_TINY,
            fill=(*RED, warn_alpha),
        )

    # Draw target trails, future ghosts, brackets
    for idx, (t, x, y) in enumerate(states):
        color = t["color"]

        # Past trail
        trail = []

        for back in range(42, 0, -6):
            tx, ty = target_state(t, i - back)
            trail.append((tx, ty))

        trail.append((x, y))

        for n, (p1, p2) in enumerate(zip(trail[:-1], trail[1:])):
            alpha = int(25 + 90 * n / max(1, len(trail)))
            d.line([p1, p2], fill=(*color, alpha), width=1)

        # Future ghost positions
        for k in range(1, 6):
            gx, gy = target_state(t, i + k * 12)
            alpha = 125 - k * 18
            r = 4 + k * 0.5

            d.ellipse(
                [gx-r, gy-r, gx+r, gy+r],
                outline=(*color, alpha),
                width=1,
            )

        # Velocity vector
        nx, ny = target_state(t, i + 6)
        vx, vy = nx - x, ny - y
        norm = max(np.hypot(vx, vy), 1)

        vx, vy = vx / norm, vy / norm

        d.line(
            (x, y, x + vx * 46, y + vy * 46),
            fill=(*GREEN, 135),
            width=1,
        )

        # Lock brackets
        size = 22 + 4 * np.sin(phase * 4 + idx)
        draw_brackets(d, x, y, size, color, alpha=220)

        d.ellipse(
            [x - 3, y - 3, x + 3, y + 3],
            fill=(*WHITE, 220),
        )

        # Label
        d.text((x + 28, y - 15), t["id"], font=FONT_TINY, fill=(*color, 210))
        d.text((x + 28, y), t["type"], font=FONT_TINY, fill=(*CYAN2, 145))

    # Right priority stack
    px, py = 35, 90
    pw, ph = 220, 330

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 100), width=1)
    d.line((px, py, px + 92, py), fill=(*CYAN2, 190), width=3)

    d.text((px + 15, py + 15), "TRACK QUEUE", font=FONT_SMALL, fill=(*CYAN2, 190))

    for idx, (t, x, y) in enumerate(states):
        yy = py + 55 + idx * 48
        color = t["color"]

        rng_val = np.hypot(x - W / 2, y - H / 2)
        priority = max(0, 100 - rng_val / 4)

        d.text((px + 16, yy), t["id"], font=FONT_TINY, fill=(*color, 210))
        d.text((px + 86, yy), f"{priority:04.1f}", font=FONT_SMALL, fill=(*CYAN2, 220))

        # priority bar
        bx = px + 16
        by = yy + 22
        bw = 160

        d.rectangle([bx, by, bx + bw, by + 7], outline=(*CYAN, 60), width=1)
        d.rectangle([bx + 2, by + 2, bx + 2 + int((bw - 4) * priority / 100), by + 5], fill=(*color, 150))

    # Footer
    d.text(
        (32, H - 30),
        "MULTI-OBJECT TRACKING • CROSSING VECTOR PREDICTION • PRIORITY SOLVER ACTIVE",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    # HUD corners
    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.7))

    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "multi_target_tracker.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/tracking/multi_target_tracker.gif


In [1]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/navigation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)


def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]

    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass

    return ImageFont.load_default()


FONT = load_font(22)
FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def bezier(p0, p1, p2, p3, t):
    return (
        (1 - t) ** 3 * p0
        + 3 * (1 - t) ** 2 * t * p1
        + 3 * (1 - t) * t ** 2 * p2
        + t ** 3 * p3
    )


MAP_X0, MAP_Y0 = 315, 72
MAP_X1, MAP_Y1 = 915, 415
MAP_W = MAP_X1 - MAP_X0
MAP_H = MAP_Y1 - MAP_Y0

STAR = np.array([MAP_X0 + 150, MAP_Y0 + 178])
WP1 = np.array([MAP_X0 + 260, MAP_Y0 + 90])
WP2 = np.array([MAP_X0 + 405, MAP_Y0 + 235])
DEST = np.array([MAP_X0 + 535, MAP_Y0 + 105])

ROUTE = np.array([
    bezier(
        STAR,
        np.array([MAP_X0 + 210, MAP_Y0 + 30]),
        np.array([MAP_X0 + 340, MAP_Y0 + 300]),
        DEST,
        t,
    )
    for t in np.linspace(0, 1, 460)
])

WAYPOINTS = [
    ("ORIGIN", STAR, CYAN2),
    ("ASSIST-1", WP1, GREEN),
    ("ASSIST-2", WP2, YELLOW),
    ("TARGET", DEST, ORANGE),
]


def draw_body(d, pos, radius, color, phase, label):
    x, y = pos
    pulse = 0.55 + 0.45 * np.sin(phase * 3 + radius)

    for rr, a in [(radius + 12, 28), (radius + 6, 45)]:
        d.ellipse(
            [x - rr, y - rr, x + rr, y + rr],
            outline=(*color, int(a * pulse)),
            width=1,
        )

    d.ellipse(
        [x - radius, y - radius, x + radius, y + radius],
        fill=(*color, 190),
    )

    d.text(
        (x + radius + 8, y - 7),
        label,
        font=FONT_TINY,
        fill=(*color, 175),
    )


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    route_progress = smoothstep((i - 18) / 105)
    lock = smoothstep((i - 120) / 35)

    # Header
    d.text((32, 10), "NAVIGATION COMPUTER", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 215, 34), fill=(*CYAN, 170), width=2)
    d.line((215, 34, 238, 48), fill=(*CYAN, 170), width=2)
    d.line((238, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Left telemetry stack
    px, py = 35, 72
    pw, ph = 240, 345

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 100), width=1)
    d.line((px, py, px + 92, py), fill=(*CYAN2, 190), width=3)

    d.text((px + 16, py + 16), "ROUTE SOLVER", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("ΔV NODE 1", f"{2.31 + 0.04*np.sin(phase):.2f} km/s"),
        ("ΔV NODE 2", f"{1.78 + 0.03*np.cos(phase):.2f} km/s"),
        ("ETA", f"T+{72 - int(i/3):03d} h"),
        ("ASSIST", "READY" if i > 65 else "CALC"),
        ("WINDOW", "OPEN" if i > 95 else "WAIT"),
        ("LOCK", f"{int(lock*100):03d}%"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 58 + m * 42

        col = GREEN if v in ("READY", "OPEN") else CYAN2

        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 112, y), v, font=FONT_SMALL, fill=(*col, 220))

        for t in range(5):
            a = int(35 + 90 * np.sin(phase * 4 + m + t) ** 2)
            tx = px + 202 + t * 6
            d.line((tx, y + 8, tx + 3, y + 8), fill=(*CYAN, a), width=1)

    # Map frame
    d.rectangle([MAP_X0, MAP_Y0, MAP_X1, MAP_Y1], outline=(*CYAN, 95), width=1)
    d.line((MAP_X0, MAP_Y0, MAP_X0 + 120, MAP_Y0), fill=(*CYAN2, 180), width=3)

    d.text((MAP_X0 + 16, MAP_Y0 + 14), "SYSTEM MAP / TRANSFER ROUTE", font=FONT_TINY, fill=(*CYAN2, 170))

    # Map grid
    for x in range(MAP_X0 + 60, MAP_X1, 60):
        d.line((x, MAP_Y0, x, MAP_Y1), fill=(*CYAN, 14), width=1)

    for y in range(MAP_Y0 + 50, MAP_Y1, 50):
        d.line((MAP_X0, y, MAP_X1, y), fill=(*CYAN, 14), width=1)

    # Orbits
    for r, a in [(72, 30), (142, 26), (215, 22)]:
        d.ellipse(
            [STAR[0] - r, STAR[1] - r * 0.55, STAR[0] + r, STAR[1] + r * 0.55],
            outline=(*CYAN, a),
            width=1,
        )

    # Bodies / waypoints
    draw_body(d, STAR, 12, WHITE, phase, "STAR")
    draw_body(d, WP1, 8, GREEN, phase, "WP1")
    draw_body(d, WP2, 9, YELLOW, phase, "WP2")
    draw_body(d, DEST, 11, ORANGE, phase, "DEST")

    # Route trail
    n = int(len(ROUTE) * route_progress)
    visible_route = ROUTE[:max(2, n)]

    for idx, (p1, p2) in enumerate(zip(visible_route[:-1], visible_route[1:])):
        a = int(70 + 100 * idx / max(1, len(visible_route)))
        d.line([tuple(p1), tuple(p2)], fill=(*CYAN2, a), width=2)

    # Future projected route after current ship
    if n > 10:
        for k in range(1, 8):
            gi = min(n + k * 18, len(ROUTE) - 1)
            gx, gy = ROUTE[gi]
            a = 130 - k * 15
            d.ellipse([gx - 4, gy - 4, gx + 4, gy + 4], outline=(*YELLOW, a), width=1)

    # Ship marker
    ship_idx = min(max(0, n - 1), len(ROUTE) - 1)
    sx, sy = ROUTE[ship_idx]

    d.ellipse([sx - 6, sy - 6, sx + 6, sy + 6], fill=(*CYAN2, 245))
    d.ellipse([sx - 18, sy - 18, sx + 18, sy + 18], outline=(*CYAN2, 100), width=2)

    # Velocity vector
    next_idx = min(ship_idx + 4, len(ROUTE) - 1)
    vx, vy = ROUTE[next_idx] - ROUTE[ship_idx]
    norm = max(np.hypot(vx, vy), 1)
    vx, vy = vx / norm, vy / norm
    d.line((sx, sy, sx + vx * 42, sy + vy * 42), fill=(*GREEN, 180), width=2)

    # Gravity assist arc
    if i > 65:
        arc_alpha = int(120 * smoothstep((i - 65) / 25))
        d.arc(
            [WP1[0] - 42, WP1[1] - 42, WP1[0] + 42, WP1[1] + 42],
            start=phase * 60,
            end=phase * 60 + 250,
            fill=(*GREEN, arc_alpha),
            width=2,
        )

    # Lock flash around destination
    if lock > 0:
        a = int(150 * lock * (0.65 + 0.35 * np.sin(phase * 5)))
        d.rectangle(
            [DEST[0] - 42, DEST[1] - 42, DEST[0] + 42, DEST[1] + 42],
            outline=(*ORANGE, a),
            width=2,
        )

    # Bottom timeline
    tx0, ty0 = 45, 460
    tx1 = 910

    d.line((tx0, ty0, tx1, ty0), fill=(*CYAN, 80), width=1)

    nodes = [
        ("BURN", 0.12, CYAN2),
        ("ASSIST", 0.36, GREEN),
        ("MIDCOURSE", 0.58, YELLOW),
        ("INSERT", 0.86, ORANGE),
    ]

    for label, t, col in nodes:
        x = tx0 + (tx1 - tx0) * t
        d.line((x, ty0 - 16, x, ty0 + 16), fill=(*col, 160), width=2)
        d.text((x - 22, ty0 + 22), label, font=FONT_TINY, fill=(*col, 145))

    cursor_x = tx0 + (tx1 - tx0) * route_progress
    d.ellipse([cursor_x - 6, ty0 - 6, cursor_x + 6, ty0 + 6], fill=(*CYAN2, 220))

    d.text(
        (32, H - 28),
        "TRANSFER SOLUTION ACTIVE • GRAVITY ASSIST NODE READY • ARRIVAL WINDOW LOCK",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    # HUD corners
    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.7))

    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "navigation_computer.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/navigation/navigation_computer.gif


In [2]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/navigation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)


def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]

    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass

    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def quad_bezier(p0, p1, p2, n=140):
    t = np.linspace(0, 1, n)
    return (
        ((1 - t) ** 2)[:, None] * p0
        + (2 * (1 - t) * t)[:, None] * p1
        + (t ** 2)[:, None] * p2
    )


def orbit_pos(center, a, b, theta):
    return np.array([
        center[0] + a * np.cos(theta),
        center[1] + b * np.sin(theta),
    ])


def draw_polyline(d, pts, color, alpha=170, width=2):
    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([tuple(p1), tuple(p2)], fill=(*color, alpha), width=width)


MAP_X0, MAP_Y0 = 315, 72
MAP_X1, MAP_Y1 = 915, 415
CENTER = np.array([MAP_X0 + 235, MAP_Y0 + 178])

ORBITS = {
    "inner":  {"a": 92,  "b": 48,  "omega": 1.45, "theta0": 1.2, "color": CYAN2, "r": 7},
    "assist": {"a": 165, "b": 86,  "omega": 0.75, "theta0": 2.4, "color": GREEN, "r": 9},
    "target": {"a": 245, "b": 128, "omega": 0.42, "theta0": 5.0, "color": ORANGE, "r": 11},
}


def draw_body(d, pos, radius, color, phase, label):
    x, y = pos
    pulse = 0.55 + 0.45 * np.sin(phase * 3 + radius)

    for rr, a in [(radius + 12, 28), (radius + 6, 45)]:
        d.ellipse(
            [x - rr, y - rr, x + rr, y + rr],
            outline=(*color, int(a * pulse)),
            width=1,
        )

    d.ellipse(
        [x - radius, y - radius, x + radius, y + radius],
        fill=(*color, 190),
    )

    d.text(
        (x + radius + 8, y - 7),
        label,
        font=FONT_TINY,
        fill=(*color, 175),
    )


def build_route(phase):
    inner = ORBITS["inner"]
    assist = ORBITS["assist"]
    target = ORBITS["target"]

    p_inner = orbit_pos(CENTER, inner["a"], inner["b"], inner["theta0"] + inner["omega"] * phase)
    p_assist = orbit_pos(CENTER, assist["a"], assist["b"], assist["theta0"] + assist["omega"] * phase)
    p_target = orbit_pos(CENTER, target["a"], target["b"], target["theta0"] + target["omega"] * phase)

    # Segment A: transfer from inner orbit to assist planet.
    ctrl_a = (p_inner + p_assist) / 2 + np.array([-30, -105])
    seg_a = quad_bezier(p_inner, ctrl_a, p_assist, n=150)

    # Segment B: flyby loop around assist planet.
    flyby = []
    start_ang = np.arctan2(seg_a[-2][1] - p_assist[1], seg_a[-2][0] - p_assist[0])
    for t in np.linspace(0, 1, 110):
        ang = start_ang + 1.45 * np.pi * t
        rr = 38 - 10 * np.sin(np.pi * t)
        flyby.append(p_assist + np.array([np.cos(ang) * rr, np.sin(ang) * rr * 0.72]))
    seg_b = np.array(flyby)

    # Segment C: outbound transfer to moving target.
    flyby_exit = seg_b[-1]
    ctrl_c = (flyby_exit + p_target) / 2 + np.array([95, -65])
    seg_c = quad_bezier(flyby_exit, ctrl_c, p_target, n=180)

    route = np.vstack([seg_a, seg_b, seg_c])

    return route, p_inner, p_assist, p_target


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    route_progress = smoothstep((i - 12) / 125)
    assist_lock = smoothstep((i - 68) / 28)
    target_lock = smoothstep((i - 125) / 35)

    route, p_inner, p_assist, p_target = build_route(phase)

    # Header
    d.text((32, 10), "NAVIGATION COMPUTER • GRAVITY ASSIST SOLUTION", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 300, 34), fill=(*CYAN, 170), width=2)
    d.line((300, 34, 323, 48), fill=(*CYAN, 170), width=2)
    d.line((323, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Left telemetry
    px, py = 35, 72
    pw, ph = 240, 345

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 100), width=1)
    d.line((px, py, px + 92, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "ROUTE SOLVER", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("DEPART", "INNER ORBIT"),
        ("ASSIST", "READY" if i > 68 else "CALC"),
        ("FLYBY", f"{int(assist_lock * 100):03d}%"),
        ("TARGET", "LOCK" if target_lock > 0.85 else "TRACK"),
        ("ΔV", f"{4.12 + 0.08*np.sin(phase):.2f} km/s"),
        ("ETA", f"T+{max(0, 84 - int(i/2)):03d} h"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 58 + m * 42
        col = GREEN if v in ("READY", "LOCK") else CYAN2

        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 104, y), v, font=FONT_SMALL, fill=(*col, 220))

        for t in range(5):
            a = int(35 + 90 * np.sin(phase * 4 + m + t) ** 2)
            tx = px + 202 + t * 6
            d.line((tx, y + 8, tx + 3, y + 8), fill=(*CYAN, a), width=1)

    # Map frame
    d.rectangle([MAP_X0, MAP_Y0, MAP_X1, MAP_Y1], outline=(*CYAN, 95), width=1)
    d.line((MAP_X0, MAP_Y0, MAP_X0 + 160, MAP_Y0), fill=(*CYAN2, 180), width=3)
    d.text((MAP_X0 + 16, MAP_Y0 + 14), "SYSTEM MAP / MOVING ORBITAL BODIES", font=FONT_TINY, fill=(*CYAN2, 170))

    # Map grid
    for x in range(MAP_X0 + 60, MAP_X1, 60):
        d.line((x, MAP_Y0, x, MAP_Y1), fill=(*CYAN, 14), width=1)
    for y in range(MAP_Y0 + 50, MAP_Y1, 50):
        d.line((MAP_X0, y, MAP_X1, y), fill=(*CYAN, 14), width=1)

    # Star
    for rr, a in [(22, 35), (14, 80)]:
        d.ellipse([CENTER[0]-rr, CENTER[1]-rr, CENTER[0]+rr, CENTER[1]+rr], outline=(*WHITE, a), width=2)
    d.ellipse([CENTER[0]-7, CENTER[1]-7, CENTER[0]+7, CENTER[1]+7], fill=(*WHITE, 245))

    # Orbits + moving planets
    for name, od in ORBITS.items():
        orbit_pts = []
        for t in np.linspace(0, 2 * np.pi, 240):
            orbit_pts.append(orbit_pos(CENTER, od["a"], od["b"], t))
        draw_polyline(d, orbit_pts, od["color"], alpha=38, width=1)
        draw_polyline(d, np.array([orbit_pts[-1], orbit_pts[0]]), od["color"], alpha=38, width=1)

    draw_body(d, p_inner, ORBITS["inner"]["r"], ORBITS["inner"]["color"], phase, "INNER")
    draw_body(d, p_assist, ORBITS["assist"]["r"], ORBITS["assist"]["color"], phase, "ASSIST")
    draw_body(d, p_target, ORBITS["target"]["r"], ORBITS["target"]["color"], phase, "TARGET")

    # Route visible part
    n = int(len(route) * route_progress)
    visible = route[:max(2, n)]

    for idx, (p1, p2) in enumerate(zip(visible[:-1], visible[1:])):
        a = int(55 + 130 * idx / max(1, len(visible)))
        d.line([tuple(p1), tuple(p2)], fill=(*CYAN2, a), width=2)

    # Full faint projected route
    for p1, p2 in zip(route[:-1:8], route[1::8]):
        d.line([tuple(p1), tuple(p2)], fill=(*CYAN, 28), width=1)

    # Ship marker
    ship_idx = min(max(0, n - 1), len(route) - 1)
    sx, sy = route[ship_idx]

    d.ellipse([sx - 6, sy - 6, sx + 6, sy + 6], fill=(*CYAN2, 245))
    d.ellipse([sx - 18, sy - 18, sx + 18, sy + 18], outline=(*CYAN2, 110), width=2)

    # Velocity vector
    next_idx = min(ship_idx + 5, len(route) - 1)
    vx, vy = route[next_idx] - route[ship_idx]
    norm = max(np.hypot(vx, vy), 1)
    vx, vy = vx / norm, vy / norm
    d.line((sx, sy, sx + vx * 42, sy + vy * 42), fill=(*GREEN, 190), width=2)

    # Gravity assist highlight
    if assist_lock > 0:
        a = int(160 * assist_lock * (0.65 + 0.35 * np.sin(phase * 5)))
        d.arc(
            [p_assist[0] - 48, p_assist[1] - 38, p_assist[0] + 48, p_assist[1] + 38],
            start=phase * 80,
            end=phase * 80 + 280,
            fill=(*GREEN, a),
            width=3,
        )
        d.text((p_assist[0] + 45, p_assist[1] + 28), "GRAV ASSIST", font=FONT_TINY, fill=(*GREEN, int(190 * assist_lock)))

    # Target lock
    if target_lock > 0:
        a = int(160 * target_lock * (0.65 + 0.35 * np.sin(phase * 5)))
        d.rectangle(
            [p_target[0] - 42, p_target[1] - 42, p_target[0] + 42, p_target[1] + 42],
            outline=(*ORANGE, a),
            width=2,
        )

    # Bottom timeline
    tx0, ty0 = 45, 460
    tx1 = 910

    d.line((tx0, ty0, tx1, ty0), fill=(*CYAN, 80), width=1)

    nodes = [
        ("BURN", 0.10, CYAN2),
        ("TRANSFER", 0.30, CYAN),
        ("FLYBY", 0.55, GREEN),
        ("INSERT", 0.86, ORANGE),
    ]

    for label, t, col in nodes:
        x = tx0 + (tx1 - tx0) * t
        d.line((x, ty0 - 16, x, ty0 + 16), fill=(*col, 160), width=2)
        d.text((x - 22, ty0 + 22), label, font=FONT_TINY, fill=(*col, 145))

    cursor_x = tx0 + (tx1 - tx0) * route_progress
    d.ellipse([cursor_x - 6, ty0 - 6, cursor_x + 6, ty0 + 6], fill=(*CYAN2, 220))

    d.text(
        (32, H - 28),
        "MOVING-BODY SOLUTION • ASSIST FLYBY LOOP • OUTBOUND TRANSFER TO TARGET ORBIT",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    # HUD corners
    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.7))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "navigation_computer_grav_assist.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/navigation/navigation_computer_grav_assist.gif


In [5]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/navigation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
ORANGE = (255, 160, 70)
WHITE = (245, 250, 255)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def quad_bezier(p0, p1, p2, n=160):
    t = np.linspace(0, 1, n)
    return ((1 - t) ** 2)[:, None] * p0 + (2 * (1 - t) * t)[:, None] * p1 + (t ** 2)[:, None] * p2


def orbit_pos(center, a, b, theta):
    return np.array([center[0] + a * np.cos(theta), center[1] + b * np.sin(theta)])


def draw_polyline(d, pts, color, alpha=170, width=2):
    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([tuple(p1), tuple(p2)], fill=(*color, alpha), width=width)


MAP_X0, MAP_Y0 = 315, 72
MAP_X1, MAP_Y1 = 915, 415
CENTER = np.array([MAP_X0 + 235, MAP_Y0 + 178])

ORBITS = {
    "inner": {"a": 92, "b": 48, "omega": 1.45, "theta0": 1.2, "color": CYAN2, "r": 7},
    "target": {"a": 245, "b": 128, "omega": 0.42, "theta0": 5.0, "color": ORANGE, "r": 11},
}


def draw_body(d, pos, radius, color, phase, label):
    x, y = pos
    pulse = 0.55 + 0.45 * np.sin(phase * 3 + radius)

    for rr, a in [(radius + 12, 28), (radius + 6, 45)]:
        d.ellipse([x - rr, y - rr, x + rr, y + rr], outline=(*color, int(a * pulse)), width=1)

    d.ellipse([x - radius, y - radius, x + radius, y + radius], fill=(*color, 190))
    d.text((x + radius + 8, y - 7), label, font=FONT_TINY, fill=(*color, 175))


def build_route(phase):
    inner = ORBITS["inner"]
    target = ORBITS["target"]

    p_inner = orbit_pos(
        CENTER,
        inner["a"],
        inner["b"],
        inner["theta0"] + inner["omega"] * phase,
    )

    p_target = orbit_pos(
        CENTER,
        target["a"],
        target["b"],
        target["theta0"] + target["omega"] * phase,
    )

    # flyby point рядом с внешней планетой, не в саму планету
    flyby_offset = np.array([-38, -42])
    p_flyby = p_target + flyby_offset

    # точка выхода за пределы карты
    escape = np.array([
        MAP_X1 + 85,
        MAP_Y0 - 35,
    ])

    ctrl_a = (p_inner + p_flyby) / 2 + np.array([45, -95])
    ctrl_b = (p_flyby + escape) / 2 + np.array([95, -28])

    seg_a = quad_bezier(p_inner, ctrl_a, p_flyby, n=220)
    seg_b = quad_bezier(p_flyby, ctrl_b, escape, n=190)

    route = np.vstack([seg_a, seg_b])

    return route, p_inner, p_target, escape, p_flyby



def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    route_progress = smoothstep((i - 12) / 130)
    target_lock = smoothstep((i - 100) / 35)
    escape_lock = smoothstep((i - 130) / 35)

    route, p_inner, p_target, escape, p_flyby = build_route(phase)

    d.text((32, 10), "NAVIGATION COMPUTER • OUTBOUND ESCAPE SOLUTION", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 300, 34), fill=(*CYAN, 170), width=2)
    d.line((300, 34, 323, 48), fill=(*CYAN, 170), width=2)
    d.line((323, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Left telemetry
    px, py = 35, 72
    pw, ph = 240, 345

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 100), width=1)
    d.line((px, py, px + 92, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "ROUTE SOLVER", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("DEPART", "INNER ORBIT"),
        ("TRANSFER", "ACTIVE"),
        ("TARGET", "PASSAGE" if i > 95 else "TRACK"),
        ("ESCAPE", "LOCK" if escape_lock > 0.85 else "CALC"),
        ("ΔV", f"{3.42 + 0.05*np.sin(phase):.2f} km/s"),
        ("ETA", f"T+{max(0, 84 - int(i / 2)):03d} h"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 58 + m * 42
        col = GREEN if v in ("ACTIVE", "LOCK", "PASSAGE") else CYAN2

        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 104, y), v, font=FONT_SMALL, fill=(*col, 220))

        for t in range(5):
            a = int(35 + 90 * np.sin(phase * 4 + m + t) ** 2)
            tx = px + 202 + t * 6
            d.line((tx, y + 8, tx + 3, y + 8), fill=(*CYAN, a), width=1)

    # Map frame
    d.rectangle([MAP_X0, MAP_Y0, MAP_X1, MAP_Y1], outline=(*CYAN, 95), width=1)
    d.line((MAP_X0, MAP_Y0, MAP_X0 + 160, MAP_Y0), fill=(*CYAN2, 180), width=3)
    d.text((MAP_X0 + 16, MAP_Y0 + 14), "SYSTEM MAP / SMOOTH OUTBOUND TRAJECTORY", font=FONT_TINY, fill=(*CYAN2, 170))

    for x in range(MAP_X0 + 60, MAP_X1, 60):
        d.line((x, MAP_Y0, x, MAP_Y1), fill=(*CYAN, 14), width=1)

    for y in range(MAP_Y0 + 50, MAP_Y1, 50):
        d.line((MAP_X0, y, MAP_X1, y), fill=(*CYAN, 14), width=1)

    # Star
    for rr, a in [(22, 35), (14, 80)]:
        d.ellipse([CENTER[0] - rr, CENTER[1] - rr, CENTER[0] + rr, CENTER[1] + rr], outline=(*WHITE, a), width=2)

    d.ellipse([CENTER[0] - 7, CENTER[1] - 7, CENTER[0] + 7, CENTER[1] + 7], fill=(*WHITE, 245))

    # Orbits and bodies
    for _, od in ORBITS.items():
        orbit_pts = [orbit_pos(CENTER, od["a"], od["b"], t) for t in np.linspace(0, 2 * np.pi, 240)]
        draw_polyline(d, orbit_pts, od["color"], alpha=38, width=1)
        draw_polyline(d, np.array([orbit_pts[-1], orbit_pts[0]]), od["color"], alpha=38, width=1)

    draw_body(d, p_inner, ORBITS["inner"]["r"], ORBITS["inner"]["color"], phase, "INNER")
    draw_body(d, p_target, ORBITS["target"]["r"], ORBITS["target"]["color"], phase, "TARGET")

    # Route
    n = int(len(route) * route_progress)
    visible = route[:max(2, n)]

    for idx, (p1, p2) in enumerate(zip(visible[:-1], visible[1:])):
        a = int(55 + 130 * idx / max(1, len(visible)))
        d.line([tuple(p1), tuple(p2)], fill=(*CYAN2, a), width=2)

    # Faint projected route
    for p1, p2 in zip(route[:-1:8], route[1::8]):
        d.line([tuple(p1), tuple(p2)], fill=(*CYAN, 28), width=1)

    # Ship marker
    ship_idx = min(max(0, n - 1), len(route) - 1)
    sx, sy = route[ship_idx]

    d.ellipse([sx - 6, sy - 6, sx + 6, sy + 6], fill=(*CYAN2, 245))
    d.ellipse([sx - 18, sy - 18, sx + 18, sy + 18], outline=(*CYAN2, 110), width=2)

    next_idx = min(ship_idx + 5, len(route) - 1)
    vx, vy = route[next_idx] - route[ship_idx]
    norm = max(np.hypot(vx, vy), 1)
    vx, vy = vx / norm, vy / norm

    d.line((sx, sy, sx + vx * 42, sy + vy * 42), fill=(*GREEN, 190), width=2)

    # Target passage
    if target_lock > 0:
        a = int(150 * target_lock * (0.65 + 0.35 * np.sin(phase * 5)))

        d.rectangle(
            [
                p_flyby[0] - 30,
                p_flyby[1] - 30,
                p_flyby[0] + 30,
                p_flyby[1] + 30,
            ],
            outline=(*ORANGE, a),
            width=2,
        )

        d.text(
            (p_flyby[0] + 36, p_flyby[1] - 8),
            "PASSAGE",
            font=FONT_TINY,
            fill=(*ORANGE, a),
        )

    # Escape vector marker
    if escape_lock > 0:
        a = int(160 * escape_lock)
        ex, ey = escape

        d.line((ex - 28, ey, ex + 28, ey), fill=(*GREEN, a), width=2)
        d.line((ex, ey - 28, ex, ey + 28), fill=(*GREEN, a), width=2)
        d.text((ex - 92, ey + 34), "ESCAPE VECTOR", font=FONT_TINY, fill=(*GREEN, a))

    # Timeline
    tx0, ty0 = 45, 460
    tx1 = 910
    d.line((tx0, ty0, tx1, ty0), fill=(*CYAN, 80), width=1)

    nodes = [
        ("BURN", 0.12, CYAN2),
        ("TRANSFER", 0.42, CYAN),
        ("TARGET", 0.72, ORANGE),
        ("ESCAPE", 0.92, GREEN),
    ]

    for label, t, col in nodes:
        x = tx0 + (tx1 - tx0) * t
        d.line((x, ty0 - 16, x, ty0 + 16), fill=(*col, 160), width=2)
        d.text((x - 22, ty0 + 22), label, font=FONT_TINY, fill=(*col, 145))

    cursor_x = tx0 + (tx1 - tx0) * route_progress
    d.ellipse([cursor_x - 6, ty0 - 6, cursor_x + 6, ty0 + 6], fill=(*CYAN2, 220))

    d.text(
        (32, H - 28),
        "SMOOTH OUTBOUND TRAJECTORY • TARGET PASSAGE • SYSTEM ESCAPE VECTOR LOCKED",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.7))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "navigation_computer_escape.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/navigation/navigation_computer_escape.gif


In [6]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/lidar")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(42)


def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]

    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass

    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


POINTS = []

for _ in range(260):
    POINTS.append({
        "x": rng.uniform(-1.2, 1.2),
        "y": rng.uniform(-0.8, 0.8),
        "z": rng.uniform(0.1, 1.0),
        "size": rng.uniform(1.0, 3.5),
        "kind": rng.choice(["dust", "rock", "metal"]),
    })


def rotate_y(x, z, ang):
    ca = np.cos(ang)
    sa = np.sin(ang)

    return (
        x * ca - z * sa,
        x * sa + z * ca
    )


def project_point(x, y, z):
    fov = 420 / (z + 1.45)

    sx = W * 0.55 + x * fov
    sy = H * 0.50 + y * fov

    return sx, sy, fov


def render_frame(i):

    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    # ---------------------------------------------------
    # Header
    # ---------------------------------------------------

    d.text(
        (32, 10),
        "DEEP SPACE LIDAR",
        font=FONT_SMALL,
        fill=(*CYAN2, 190),
    )

    d.line((28, 34, 185, 34), fill=(*CYAN, 170), width=2)
    d.line((185, 34, 208, 48), fill=(*CYAN, 170), width=2)
    d.line((208, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # ---------------------------------------------------
    # Main scan frame
    # ---------------------------------------------------

    frame = [55, 72, 735, 440]

    d.rectangle(
        frame,
        outline=(*CYAN, 90),
        width=1,
    )

    d.line(
        (55, 72, 175, 72),
        fill=(*CYAN2, 190),
        width=3,
    )

    # ---------------------------------------------------
    # Volumetric grid
    # ---------------------------------------------------

    for x in range(85, 735, 48):
        d.line((x, 72, x, 440), fill=(*CYAN, 10), width=1)

    for y in range(100, 440, 42):
        d.line((55, y, 735, y), fill=(*CYAN, 10), width=1)

    # ---------------------------------------------------
    # Rotating scan beam
    # ---------------------------------------------------

    beam_x = 90 + (i * 5.8) % 620

    for dx in range(-16, 17):
        a = max(0, 42 - abs(dx) * 2)

        d.line(
            (
                beam_x + dx,
                76,
                beam_x + dx,
                436
            ),
            fill=(*CYAN2, a),
            width=1,
        )

    # ---------------------------------------------------
    # Point cloud
    # ---------------------------------------------------

    visible_pts = []

    for idx, p in enumerate(POINTS):

        rx, rz = rotate_y(
            p["x"],
            p["z"],
            phase * 0.32,
        )

        sx, sy, scale = project_point(
            rx,
            p["y"],
            rz,
        )

        if not (60 <= sx <= 730 and 76 <= sy <= 436):
            continue

        visible_pts.append((sx, sy, rz))

        depth_alpha = int(210 / (rz + 1.1))

        # beam highlight
        beam_dist = abs(sx - beam_x)
        scan_boost = np.exp(-(beam_dist ** 2) / 900)

        alpha = min(
            255,
            int(depth_alpha + scan_boost * 140)
        )

        rr = p["size"] * scale * 0.012

        if p["kind"] == "rock":
            col = CYAN

        elif p["kind"] == "metal":
            col = WHITE

        else:
            col = CYAN2

        # glow
        d.ellipse(
            [
                sx - rr * 2.2,
                sy - rr * 2.2,
                sx + rr * 2.2,
                sy + rr * 2.2
            ],
            fill=(*col, int(alpha * 0.16)),
        )

        # core
        d.ellipse(
            [
                sx - rr,
                sy - rr,
                sx + rr,
                sy + rr
            ],
            fill=(*col, alpha),
        )

        # object brackets for larger echoes
        if rr > 2.2 and scan_boost > 0.55:

            b = rr * 4

            d.line(
                (sx - b, sy - b, sx - b/2, sy - b),
                fill=(*GREEN, 180),
                width=1,
            )

            d.line(
                (sx - b, sy - b, sx - b, sy - b/2),
                fill=(*GREEN, 180),
                width=1,
            )

            d.line(
                (sx + b, sy + b, sx + b/2, sy + b),
                fill=(*GREEN, 180),
                width=1,
            )

            d.line(
                (sx + b, sy + b, sx + b, sy + b/2),
                fill=(*GREEN, 180),
                width=1,
            )

    # ---------------------------------------------------
    # Depth fog
    # ---------------------------------------------------

    fog = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    fd = ImageDraw.Draw(fog)

    for k in range(6):

        yy = 80 + k * 65
        aa = 12 + 8 * np.sin(phase * 2 + k)

        fd.rectangle(
            [56, yy, 734, yy + 38],
            fill=(0, 20, 35, int(aa)),
        )

    img.alpha_composite(fog)

    # ---------------------------------------------------
    # Right telemetry panel
    # ---------------------------------------------------

    px, py = 770, 82
    pw, ph = 155, 320

    d.rectangle(
        [px, py, px + pw, py + ph],
        outline=(*CYAN, 100),
        width=1,
    )

    d.line(
        (px, py, px + 75, py),
        fill=(*CYAN2, 190),
        width=3,
    )

    d.text(
        (px + 12, py + 14),
        "SCAN DATA",
        font=FONT_SMALL,
        fill=(*CYAN2, 190),
    )

    closest = sorted(
        visible_pts,
        key=lambda v: v[2]
    )[:5]

    metrics = [
        ("ECHOES", f"{len(visible_pts):03d}"),
        ("SCAN", "ACTIVE"),
        ("DEPTH", "12.4 AU"),
        ("FIELD", "DENSE"),
        ("TRACK", f"{len(closest):02d}"),
    ]

    for idx, (k, v) in enumerate(metrics):

        yy = py + 54 + idx * 42

        col = GREEN if v == "ACTIVE" else CYAN2

        d.text(
            (px + 12, yy),
            k,
            font=FONT_TINY,
            fill=(*CYAN, 150),
        )

        d.text(
            (px + 76, yy),
            v,
            font=FONT_SMALL,
            fill=(*col, 220),
        )

    # ---------------------------------------------------
    # Mini depth bars
    # ---------------------------------------------------

    bx = px + 18
    by = py + 250

    for idx in range(12):

        hh = 12 + 45 * (
            0.5 + 0.5 * np.sin(
                phase * 3
                + idx * 0.8
            )
        )

        xx = bx + idx * 10

        d.rectangle(
            [xx, by - hh, xx + 5, by],
            fill=(*CYAN2, 160),
        )

    # ---------------------------------------------------
    # Target labels
    # ---------------------------------------------------

    for idx, (sx, sy, rz) in enumerate(closest[:3]):

        d.text(
            (
                sx + 12,
                sy - 10
            ),
            f"ECHO-{idx+1}",
            font=FONT_TINY,
            fill=(*GREEN, 180),
        )

    # ---------------------------------------------------
    # Footer
    # ---------------------------------------------------

    d.text(
        (32, H - 28),
        "POINT CLOUD RECONSTRUCTION • VOLUMETRIC SWEEP • DEEP SPACE ECHO ANALYSIS",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    # ---------------------------------------------------
    # HUD corners
    # ---------------------------------------------------

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)

    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    # ---------------------------------------------------
    # Glow
    # ---------------------------------------------------

    glow = img.filter(
        ImageFilter.GaussianBlur(1.8)
    )

    final = Image.new("RGBA", (W, H), BG)

    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [
    render_frame(i)
    for i in range(FRAMES)
]

OUT = OUT_DIR / "deep_space_lidar.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/lidar/deep_space_lidar.gif


In [7]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/lidar")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 540, 960
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
WHITE = (245, 250, 255)

rng = np.random.default_rng(42)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


POINTS = []
for _ in range(260):
    POINTS.append({
        "x": rng.uniform(-1.2, 1.2),
        "y": rng.uniform(-0.8, 0.8),
        "z": rng.uniform(0.1, 1.0),
        "size": rng.uniform(1.0, 3.5),
        "kind": rng.choice(["dust", "rock", "metal"]),
    })


def rotate_y(x, z, ang):
    ca = np.cos(ang)
    sa = np.sin(ang)
    return x * ca - z * sa, x * sa + z * ca


def project_point(x, y, z, frame):
    x0, y0, x1, y1 = frame
    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2
    fov = 300 / (z + 1.45)

    return cx + x * fov, cy + y * fov, fov


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    # Header
    d.text((28, 12), "DEEP SPACE LIDAR", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((24, 38, 180, 38), fill=(*CYAN, 170), width=2)
    d.line((180, 38, 204, 52), fill=(*CYAN, 170), width=2)
    d.line((204, 52, W - 28, 52), fill=(*CYAN, 75), width=1)

    # Lidar frame, top
    scan_frame = [36, 78, W - 36, 560]
    x0, y0, x1, y1 = scan_frame

    d.rectangle(scan_frame, outline=(*CYAN, 90), width=1)
    d.line((x0, y0, x0 + 130, y0), fill=(*CYAN2, 190), width=3)

    # Grid
    for x in range(x0 + 36, x1, 40):
        d.line((x, y0, x, y1), fill=(*CYAN, 10), width=1)

    for y in range(y0 + 40, y1, 40):
        d.line((x0, y, x1, y), fill=(*CYAN, 10), width=1)

    # Scan beam
    beam_x = x0 + 28 + (i * 4.2) % ((x1 - x0) - 56)

    for dx in range(-14, 15):
        a = max(0, 38 - abs(dx) * 2)
        d.line((beam_x + dx, y0 + 4, beam_x + dx, y1 - 4), fill=(*CYAN2, a), width=1)

    visible_pts = []

    for idx, p in enumerate(POINTS):
        rx, rz = rotate_y(p["x"], p["z"], phase * 0.32)
        sx, sy, scale = project_point(rx, p["y"], rz, scan_frame)

        if not (x0 + 5 <= sx <= x1 - 5 and y0 + 5 <= sy <= y1 - 5):
            continue

        visible_pts.append((sx, sy, rz))

        depth_alpha = int(210 / (rz + 1.1))
        beam_dist = abs(sx - beam_x)
        scan_boost = np.exp(-(beam_dist ** 2) / 700)

        alpha = min(255, int(depth_alpha + scan_boost * 140))
        rr = p["size"] * scale * 0.012

        col = WHITE if p["kind"] == "metal" else CYAN2 if p["kind"] == "dust" else CYAN

        d.ellipse([sx - rr * 2.2, sy - rr * 2.2, sx + rr * 2.2, sy + rr * 2.2], fill=(*col, int(alpha * 0.16)))
        d.ellipse([sx - rr, sy - rr, sx + rr, sy + rr], fill=(*col, alpha))

        if rr > 2.0 and scan_boost > 0.55:
            b = rr * 4
            d.line((sx - b, sy - b, sx - b / 2, sy - b), fill=(*GREEN, 180), width=1)
            d.line((sx - b, sy - b, sx - b, sy - b / 2), fill=(*GREEN, 180), width=1)
            d.line((sx + b, sy + b, sx + b / 2, sy + b), fill=(*GREEN, 180), width=1)
            d.line((sx + b, sy + b, sx + b, sy + b / 2), fill=(*GREEN, 180), width=1)

    # Depth fog
    fog = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    fd = ImageDraw.Draw(fog)

    for k in range(7):
        yy = y0 + 8 + k * 66
        aa = int(10 + 8 * np.sin(phase * 2 + k))
        fd.rectangle([x0 + 1, yy, x1 - 1, yy + 40], fill=(0, 20, 35, aa))

    img.alpha_composite(fog)

    # Telemetry frame, bottom
    px, py = 36, 590
    pw, ph = W - 72, 300

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 100), width=1)
    d.line((px, py, px + 110, py), fill=(*CYAN2, 190), width=3)

    d.text((px + 16, py + 16), "SCAN TELEMETRY", font=FONT_SMALL, fill=(*CYAN2, 190))

    closest = sorted(visible_pts, key=lambda v: v[2])[:5]

    metrics = [
        ("ECHOES", f"{len(visible_pts):03d}"),
        ("SCAN", "ACTIVE"),
        ("DEPTH", "12.4 AU"),
        ("FIELD", "DENSE"),
        ("TRACK", f"{len(closest):02d}"),
        ("MODE", "VOLUME"),
    ]

    for idx, (k, v) in enumerate(metrics):
        yy = py + 56 + idx * 34
        col = GREEN if v == "ACTIVE" else CYAN2

        d.text((px + 18, yy), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 108, yy), v, font=FONT_SMALL, fill=(*col, 220))

        for t in range(8):
            a = int(35 + 90 * np.sin(phase * 4 + idx + t) ** 2)
            tx = px + 250 + t * 12
            d.line((tx, yy + 8, tx + 7, yy + 8), fill=(*CYAN, a), width=1)

    # Depth bars
    bx = px + 18
    by = py + ph - 30

    for idx in range(32):
        hh = 10 + 52 * (0.5 + 0.5 * np.sin(phase * 3 + idx * 0.45))
        xx = bx + idx * 13
        d.rectangle([xx, by - hh, xx + 6, by], fill=(*CYAN2, 145))

    # Closest echo labels
    for idx, (sx, sy, rz) in enumerate(closest[:3]):
        d.text((sx + 10, sy - 10), f"ECHO-{idx + 1}", font=FONT_TINY, fill=(*GREEN, 180))

    # Footer
    d.text(
        (28, H - 34),
        "POINT CLOUD RECONSTRUCTION • VOLUMETRIC SWEEP • DEEP SPACE ECHO ANALYSIS",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    # HUD corners
    d.line((0, 0, 40, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 40, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.8))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "deep_space_lidar_vertical.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/lidar/deep_space_lidar_vertical.gif


In [9]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/reactor")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    cx, cy = 640, 270

    instability = np.exp(-0.5 * ((i - 104) / 10) ** 2)
    load = 0.62 + 0.18 * np.sin(phase * 1.2) + 0.22 * instability
    temp = 0.54 + 0.22 * np.sin(phase * 1.7) ** 2 + 0.28 * instability

    d.text((32, 10), "REACTOR CORE CONTROL", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 225, 34), fill=(*CYAN, 170), width=2)
    d.line((225, 34, 248, 48), fill=(*CYAN, 170), width=2)
    d.line((248, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Containment rings
    for r, a, w in [(210, 38, 1), (165, 55, 1), (120, 75, 2), (76, 105, 2)]:
        rr = r + 4 * np.sin(phase * 2 + r)
        d.ellipse([cx-rr, cy-rr, cx+rr, cy+rr], outline=(*CYAN, a), width=w)

    # Rotating magnetic sectors
    for k in range(18):
        start = phase * 55 + k * 20
        radius = 188 if k % 2 else 145
        col = CYAN2 if k % 3 else GREEN
        alpha = int(55 + 95 * np.sin(phase * 3 + k) ** 2)

        d.arc(
            [cx-radius, cy-radius, cx+radius, cy+radius],
            start=start,
            end=start + 9,
            fill=(*col, alpha),
            width=4 if k % 2 else 3,
        )

    # Core glow layer
    core = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    cd = ImageDraw.Draw(core)

    core_pulse = 0.68 + 0.32 * np.sin(phase * 2.8)
    if instability > 0.08:
        core_col = ORANGE
    else:
        core_col = CYAN2

    for r, a in [(88, 22), (62, 48), (38, 95), (18, 210)]:
        cd.ellipse(
            [cx-r, cy-r, cx+r, cy+r],
            fill=(*core_col, int(a * core_pulse)),
        )

    core = core.filter(ImageFilter.GaussianBlur(6))
    img.alpha_composite(core)

    d.ellipse([cx-22, cy-22, cx+22, cy+22], fill=(*WHITE, 230))
    d.ellipse([cx-42, cy-42, cx+42, cy+42], outline=(*CYAN2, 140), width=2)

    # Plasma spokes
    for k in range(24):
        ang = phase * 1.4 + k * 2 * np.pi / 24
        r0 = 32
        r1 = 190 + 10 * np.sin(phase * 2 + k)

        x0 = cx + np.cos(ang) * r0
        y0 = cy + np.sin(ang) * r0
        x1 = cx + np.cos(ang) * r1
        y1 = cy + np.sin(ang) * r1

        a = int(28 + 75 * np.sin(phase * 4 + k) ** 2)
        d.line([x0, y0, x1, y1], fill=(*CYAN, a), width=1)

    # Instability warning
    if instability > 0.05:
        a = int(180 * instability)
        d.ellipse([cx-225, cy-225, cx+225, cy+225], outline=(*RED, a), width=4)
        d.text((cx - 94, cy - 242), "CORE INSTABILITY", font=FONT_SMALL, fill=(*RED, min(255, a + 40)))

    # Right control panel
    px, py = 35, 92
    pw, ph = 245, 335

    d.rectangle([px, py, px+pw, py+ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px+96, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "CORE STATE", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("LOAD", f"{load*100:04.1f}%"),
        ("TEMP", f"{temp*100:04.1f}%"),
        ("FIELD", f"{88 + 7*np.sin(phase):04.1f}%"),
        ("SYNC", f"{97 + 2*np.sin(phase*2.3):04.1f}%"),
        ("VENT", "ARMED" if instability > 0.35 else "STBY"),
        ("STATE", "WARN" if instability > 0.35 else "NOMINAL"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 42
        col = RED if v == "WARN" else (YELLOW if v == "ARMED" else CYAN2)

        d.text((px + 18, y), k, font=FONT_TINY, fill=(*CYAN, 155))
        d.text((px + 95, y), v, font=FONT_SMALL, fill=(*col, 225))

        # load bars
        for t in range(7):
            a = int(40 + 95 * np.sin(phase * 4 + m + t) ** 2)
            tx = px + 190 + t * 6
            d.line((tx, y + 8, tx + 3, y + 8), fill=(*CYAN, a), width=1)

    # Bottom thermal strips
    bx, by = 70, 470
    for idx, (label, val, col) in enumerate([
        ("THERMAL", temp, ORANGE if temp > 0.78 else CYAN2),
        ("MAG FIELD", load, GREEN),
        ("VENT FLOW", 0.35 + 0.45 * instability, YELLOW),
    ]):
        y = by + idx * 18
        d.text((bx, y - 6), label, font=FONT_TINY, fill=(*CYAN, 135))
        d.rectangle([bx + 92, y, bx + 420, y + 8], outline=(*CYAN, 55), width=1)
        d.rectangle([bx + 94, y + 2, bx + 94 + int(324 * min(1, val)), y + 6], fill=(*col, 150))

    d.text(
        (32, H - 28),
        "CONTAINMENT FIELD ACTIVE • MAGNETIC SECTOR ROTATION • THERMAL LOAD MONITOR",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.7))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "reactor_core_control.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/reactor/reactor_core_control.gif


In [10]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/reactor")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 540, 960
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    cx, cy = W // 2, 325

    instability = np.exp(-0.5 * ((i - 104) / 10) ** 2)
    load = 0.62 + 0.18 * np.sin(phase * 1.2) + 0.22 * instability
    temp = 0.54 + 0.22 * np.sin(phase * 1.7) ** 2 + 0.28 * instability

    d.text((28, 12), "REACTOR CORE CONTROL", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((24, 38, 218, 38), fill=(*CYAN, 170), width=2)
    d.line((218, 38, 242, 52), fill=(*CYAN, 170), width=2)
    d.line((242, 52, W - 28, 52), fill=(*CYAN, 75), width=1)

    # Reactor frame
    rx0, ry0, rx1, ry1 = 36, 78, W - 36, 585
    d.rectangle([rx0, ry0, rx1, ry1], outline=(*CYAN, 85), width=1)
    d.line((rx0, ry0, rx0 + 130, ry0), fill=(*CYAN2, 190), width=3)

    # Containment rings
    for r, a, w in [(190, 32, 1), (148, 48, 1), (108, 70, 2), (68, 100, 2)]:
        rr = r + 4 * np.sin(phase * 2 + r)
        d.ellipse([cx - rr, cy - rr, cx + rr, cy + rr], outline=(*CYAN, a), width=w)

    # Rotating magnetic sectors
    for k in range(18):
        start = phase * 55 + k * 20
        radius = 172 if k % 2 else 132
        col = CYAN2 if k % 3 else GREEN
        alpha = int(55 + 95 * np.sin(phase * 3 + k) ** 2)

        d.arc(
            [cx - radius, cy - radius, cx + radius, cy + radius],
            start=start,
            end=start + 9,
            fill=(*col, alpha),
            width=4 if k % 2 else 3,
        )

    # Core glow
    core = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    cd = ImageDraw.Draw(core)

    core_pulse = 0.68 + 0.32 * np.sin(phase * 2.8)
    core_col = ORANGE if instability > 0.08 else CYAN2

    for r, a in [(78, 22), (54, 48), (32, 95), (16, 210)]:
        cd.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(*core_col, int(a * core_pulse)))

    core = core.filter(ImageFilter.GaussianBlur(6))
    img.alpha_composite(core)

    d.ellipse([cx - 20, cy - 20, cx + 20, cy + 20], fill=(*WHITE, 230))
    d.ellipse([cx - 38, cy - 38, cx + 38, cy + 38], outline=(*CYAN2, 140), width=2)

    # Plasma spokes
    for k in range(24):
        ang = phase * 1.4 + k * 2 * np.pi / 24
        r0 = 30
        r1 = 174 + 10 * np.sin(phase * 2 + k)

        x0 = cx + np.cos(ang) * r0
        y0 = cy + np.sin(ang) * r0
        x1 = cx + np.cos(ang) * r1
        y1 = cy + np.sin(ang) * r1

        a = int(28 + 75 * np.sin(phase * 4 + k) ** 2)
        d.line([x0, y0, x1, y1], fill=(*CYAN, a), width=1)

    if instability > 0.05:
        a = int(180 * instability)
        d.ellipse([cx - 208, cy - 208, cx + 208, cy + 208], outline=(*RED, a), width=4)
        d.text((cx - 92, cy - 226), "CORE INSTABILITY", font=FONT_SMALL, fill=(*RED, min(255, a + 40)))

    # Metrics panel below
    px, py = 36, 620
    pw, ph = W - 72, 250

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 120, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "CORE STATE", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("LOAD", f"{load * 100:04.1f}%"),
        ("TEMP", f"{temp * 100:04.1f}%"),
        ("FIELD", f"{88 + 7 * np.sin(phase):04.1f}%"),
        ("SYNC", f"{97 + 2 * np.sin(phase * 2.3):04.1f}%"),
        ("VENT", "ARMED" if instability > 0.35 else "STBY"),
        ("STATE", "WARN" if instability > 0.35 else "NOMINAL"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 54 + m * 28
        col = RED if v == "WARN" else (YELLOW if v == "ARMED" else CYAN2)

        d.text((px + 18, y), k, font=FONT_TINY, fill=(*CYAN, 155))
        d.text((px + 105, y), v, font=FONT_SMALL, fill=(*col, 225))

        for t in range(10):
            a = int(35 + 90 * np.sin(phase * 4 + m + t) ** 2)
            tx = px + 275 + t * 13
            d.line((tx, y + 8, tx + 8, y + 8), fill=(*CYAN, a), width=1)

    # Bottom thermal strips
    bx, by = px + 18, py + ph - 54

    for idx, (label, val, col) in enumerate([
        ("THERMAL", temp, ORANGE if temp > 0.78 else CYAN2),
        ("MAG FIELD", load, GREEN),
        ("VENT FLOW", 0.35 + 0.45 * instability, YELLOW),
    ]):
        y = by + idx * 16
        d.text((bx, y - 5), label, font=FONT_TINY, fill=(*CYAN, 135))
        d.rectangle([bx + 92, y, bx + 395, y + 7], outline=(*CYAN, 55), width=1)
        d.rectangle([bx + 94, y + 2, bx + 94 + int(299 * min(1, val)), y + 5], fill=(*col, 150))

    d.text(
        (28, H - 34),
        "CONTAINMENT FIELD ACTIVE • THERMAL LOAD MONITOR",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 40, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 40, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.7))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "reactor_core_control_vertical.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/reactor/reactor_core_control_vertical.gif


In [11]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/jump_drive")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
WHITE = (245, 250, 255)

rng = np.random.default_rng(508)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


STARS = []
for _ in range(320):
    ang = rng.uniform(0, 2 * np.pi)
    r = rng.uniform(0.02, 1.0) ** 0.65
    STARS.append({
        "x": np.cos(ang) * r,
        "y": np.sin(ang) * r,
        "z": rng.uniform(0.2, 1.0),
        "size": rng.uniform(0.7, 2.2),
        "phase": rng.uniform(0, 2 * np.pi),
    })


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    charge = smoothstep((i - 20) / 65)
    stretch = smoothstep((i - 75) / 55)
    flash = np.exp(-0.5 * ((i - 142) / 7) ** 2)
    collapse = smoothstep((i - 148) / 25)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    cx, cy = W // 2, H // 2

    d.text((32, 10), "STARFIELD JUMP DRIVE", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 220, 34), fill=(*CYAN, 170), width=2)
    d.line((220, 34, 244, 48), fill=(*CYAN, 170), width=2)
    d.line((244, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # alignment rings
    for r, a in [(70, 50), (130, 40), (210, 30), (300, 20)]:
        rr = r + 8 * np.sin(phase * 2 + r) * charge
        d.ellipse([cx - rr, cy - rr, cx + rr, cy + rr], outline=(*CYAN, a), width=1)

    # convergence geometry
    for k in range(18):
        ang = k * 2 * np.pi / 18 + phase * 0.08
        r0 = 42
        r1 = 370
        x0 = cx + np.cos(ang) * r0
        y0 = cy + np.sin(ang) * r0
        x1 = cx + np.cos(ang) * r1
        y1 = cy + np.sin(ang) * r1

        a = int((20 + 45 * np.sin(phase * 3 + k) ** 2) * charge)
        d.line((x0, y0, x1, y1), fill=(*CYAN, a), width=1)

    # stars / streaks
    for s in STARS:
        x = cx + s["x"] * W * 0.48
        y = cy + s["y"] * H * 0.82

        dx = x - cx
        dy = y - cy
        norm = max(np.hypot(dx, dy), 1)
        ux, uy = dx / norm, dy / norm

        base_alpha = int(90 + 130 * s["z"])
        size = s["size"]

        streak_len = (8 + 125 * stretch) * s["z"]
        jitter = 2 * np.sin(phase * 4 + s["phase"]) * (1 - stretch)

        if stretch > 0.05:
            x2 = x + ux * streak_len
            y2 = y + uy * streak_len
            d.line(
                (x - ux * 4, y - uy * 4 + jitter, x2, y2 + jitter),
                fill=(*WHITE, min(255, base_alpha + int(60 * stretch))),
                width=max(1, int(size)),
            )
        else:
            r = size * (0.8 + 0.5 * np.sin(phase * 2 + s["phase"]) ** 2)
            d.ellipse([x - r, y - r, x + r, y + r], fill=(*WHITE, base_alpha))

    # jump corridor
    corridor_a = int(130 * stretch * (1 - 0.5 * collapse))
    for r in [52, 82, 118, 160]:
        d.ellipse(
            [cx - r * 1.55, cy - r, cx + r * 1.55, cy + r],
            outline=(*CYAN2, corridor_a // 2),
            width=2,
        )

    # central aperture
    aperture = 22 + 64 * stretch
    d.ellipse(
        [cx - aperture, cy - aperture, cx + aperture, cy + aperture],
        outline=(*CYAN2, int(80 + 120 * stretch)),
        width=3,
    )

    d.ellipse(
        [cx - 10, cy - 10, cx + 10, cy + 10],
        fill=(*CYAN2, int(120 + 100 * charge)),
    )

    # telemetry panel
    px, py = 690, 92
    pw, ph = 225, 265

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 100), width=1)
    d.line((px, py, px + 95, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "JUMP SOLVER", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("ALIGN", f"{charge*100:04.1f}%"),
        ("FIELD", f"{stretch*100:04.1f}%"),
        ("VECTOR", "LOCK" if stretch > 0.72 else "CALC"),
        ("CORE", "CHARGED" if charge > 0.9 else "RAMP"),
        ("EXIT", "STABLE" if i > 130 else "PENDING"),
    ]

    for m, (k, v) in enumerate(metrics):
        yy = py + 56 + m * 38
        col = GREEN if v in ("LOCK", "CHARGED", "STABLE") else CYAN2
        d.text((px + 16, yy), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 94, yy), v, font=FONT_SMALL, fill=(*col, 220))

    # flash layer
    if flash > 0.02:
        a = int(210 * flash)
        d.rectangle([0, 0, W, H], fill=(*WHITE, a))
        d.text((cx - 82, cy + 120), "JUMP COMMIT", font=FONT_SMALL, fill=(*CYAN2, min(255, a + 40)))

    d.text(
        (32, H - 28),
        "NAVIGATION ALIGNMENT • FIELD CHARGE • JUMP CORRIDOR FORMATION",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.5))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "starfield_jump_drive.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/jump_drive/starfield_jump_drive.gif


In [12]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/signals")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(909)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    detect = smoothstep((i - 55) / 55)
    lock = smoothstep((i - 110) / 40)

    d.text((32, 10), "EXO SIGNAL ANALYZER", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 220, 34), fill=(*CYAN, 170), width=2)
    d.line((220, 34, 244, 48), fill=(*CYAN, 170), width=2)
    d.line((244, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Waterfall frame
    wx0, wy0 = 55, 74
    wx1, wy1 = 650, 430

    d.rectangle([wx0, wy0, wx1, wy1], outline=(*CYAN, 90), width=1)
    d.line((wx0, wy0, wx0 + 135, wy0), fill=(*CYAN2, 190), width=3)
    d.text((wx0 + 14, wy0 + 14), "WATERFALL / RF BAND", font=FONT_TINY, fill=(*CYAN2, 160))

    # Background spectral bins
    rows = 72
    cols = 112
    cell_w = (wx1 - wx0 - 24) / cols
    cell_h = (wy1 - wy0 - 42) / rows

    base_x = wx0 + 12
    base_y = wy0 + 36

    carrier_col = int(cols * (0.54 + 0.035 * np.sin(phase * 0.9)))

    for r in range(rows):
        y = base_y + r * cell_h

        time_wave = np.sin(phase * 2.4 + r * 0.28)
        pulse = np.exp(-0.5 * ((r - ((i * 0.55) % rows)) / 4.0) ** 2)

        for c in range(cols):
            x = base_x + c * cell_w

            noise = 0.10 + 0.20 * rng.random()
            carrier = np.exp(-0.5 * ((c - carrier_col - 3 * time_wave) / 2.5) ** 2)
            harmonic = 0.45 * np.exp(-0.5 * ((c - carrier_col * 0.68) / 2.2) ** 2)
            burst = pulse * np.exp(-0.5 * ((c - carrier_col) / 7.0) ** 2)

            val = noise + detect * (0.85 * carrier + 0.42 * harmonic + 1.1 * burst)
            val = min(1.0, val)

            if val < 0.22:
                continue

            if val > 0.78:
                color = YELLOW
            elif val > 0.48:
                color = CYAN2
            else:
                color = CYAN

            alpha = int(35 + 175 * val)

            d.rectangle(
                [x, y, x + cell_w + 0.5, y + cell_h + 0.5],
                fill=(*color, alpha),
            )

    # FFT line panel
    fx0, fy0 = 55, 452
    fx1, fy1 = 650, 505

    d.rectangle([fx0, fy0, fx1, fy1], outline=(*CYAN, 65), width=1)

    xs = np.linspace(fx0 + 8, fx1 - 8, 520)
    spec = (
        0.18 * np.sin(xs * 0.05 + phase)
        + 0.08 * np.sin(xs * 0.17 - phase * 1.4)
    )

    peak_x = fx0 + 8 + (fx1 - fx0 - 16) * (carrier_col / cols)
    spec += 1.2 * detect * np.exp(-0.5 * ((xs - peak_x) / 10) ** 2)

    pts = []
    for x, val in zip(xs, spec):
        y = fy1 - 12 - np.clip(val + 0.35, 0, 1.4) * 26
        pts.append((x, y))

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*CYAN2, 210), width=1)

    # Detection marker
    if detect > 0.25:
        mx = peak_x
        a = int(160 * detect * (0.65 + 0.35 * np.sin(phase * 6)))

        d.line((mx, wy0 + 36, mx, wy1 - 8), fill=(*YELLOW, a), width=1)
        d.text((mx + 12, wy0 + 48), "CARRIER", font=FONT_TINY, fill=(*YELLOW, a))

    # Right analysis panel
    px, py = 690, 92
    pw, ph = 230, 330

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 96, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "SIGNAL FIT", font=FONT_SMALL, fill=(*CYAN2, 190))

    snr = 6 + 28 * detect + 12 * lock + 2 * np.sin(phase * 1.7)
    drift = 0.004 * np.sin(phase * 1.1)
    period = 2.73 + 0.03 * np.sin(phase * 0.8)

    metrics = [
        ("SNR", f"{snr:04.1f} dB"),
        ("DRIFT", f"{drift:+.4f}"),
        ("PERIOD", f"{period:.2f} s"),
        ("CARRIER", f"{1420.4 + 0.2*np.sin(phase):.1f} MHz"),
        ("LOCK", f"{lock*100:04.1f}%"),
        ("STATE", "PATTERN" if lock > 0.75 else "SCAN"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 40
        col = GREEN if v == "PATTERN" else CYAN2
        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 94, y), v, font=FONT_SMALL, fill=(*col, 220))

    # Pulse train
    tx, ty = px + 18, py + ph - 38
    for k in range(26):
        a = int(45 + 115 * np.sin(phase * 5 + k * 0.9) ** 2)
        h = 6 + 22 * np.exp(-0.5 * ((k % 7) - 2) ** 2 / 1.2)
        d.line((tx + k * 7, ty, tx + k * 7, ty - h), fill=(*GREEN, a), width=2)

    d.text(
        (32, H - 28),
        "WATERFALL SPECTRUM • DRIFTING CARRIER • REPEATING PULSE TRAIN • PATTERN LOCK",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.4))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "exo_signal_analyzer.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/signals/exo_signal_analyzer.gif


In [13]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/swarm")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(1204)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


N = 54
SWARM = []

for idx in range(N):
    group = idx % 3
    SWARM.append({
        "id": idx,
        "group": group,
        "phase": rng.uniform(0, 2 * np.pi),
        "radius": rng.uniform(18, 150),
        "angle": rng.uniform(0, 2 * np.pi),
        "speed": rng.uniform(0.35, 1.25),
    })


def formation_center(group, phase):
    centers = [
        np.array([330 + 28 * np.sin(phase * 0.7), 240 + 18 * np.cos(phase * 0.9)]),
        np.array([575 + 34 * np.cos(phase * 0.6), 185 + 24 * np.sin(phase * 0.8)]),
        np.array([610 + 26 * np.sin(phase * 0.5), 365 + 18 * np.cos(phase * 0.7)]),
    ]
    return centers[group]


def drone_pos(drone, phase):
    center = formation_center(drone["group"], phase)
    a = drone["angle"] + phase * drone["speed"]

    if drone["group"] == 0:
        x = center[0] + np.cos(a) * drone["radius"] * 0.75
        y = center[1] + np.sin(a) * drone["radius"] * 0.45
    elif drone["group"] == 1:
        lane = (drone["id"] % 9) - 4
        x = center[0] + lane * 22 + 18 * np.sin(phase * 1.7 + drone["phase"])
        y = center[1] + (drone["id"] // 9) * 18 + 22 * np.cos(phase * 1.1 + drone["phase"])
    else:
        x = center[0] + np.cos(a) * drone["radius"] * 0.55
        y = center[1] + np.sin(a * 1.6) * drone["radius"] * 0.38

    return np.array([x, y])


def draw_bracket(d, x, y, s, color, alpha):
    c = s * 0.4
    d.line((x-s, y-s, x-s+c, y-s), fill=(*color, alpha), width=1)
    d.line((x-s, y-s, x-s, y-s+c), fill=(*color, alpha), width=1)
    d.line((x+s, y-s, x+s-c, y-s), fill=(*color, alpha), width=1)
    d.line((x+s, y-s, x+s, y-s+c), fill=(*color, alpha), width=1)
    d.line((x-s, y+s, x-s+c, y+s), fill=(*color, alpha), width=1)
    d.line((x-s, y+s, x-s, y+s-c), fill=(*color, alpha), width=1)
    d.line((x+s, y+s, x+s-c, y+s), fill=(*color, alpha), width=1)
    d.line((x+s, y+s, x+s, y+s-c), fill=(*color, alpha), width=1)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    alert = np.exp(-0.5 * ((i - 112) / 11) ** 2)

    d.text((32, 10), "DRONE SWARM TACTICAL", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 235, 34), fill=(*CYAN, 170), width=2)
    d.line((235, 34, 258, 48), fill=(*CYAN, 170), width=2)
    d.line((258, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Tactical field
    fx0, fy0, fx1, fy1 = 42, 72, 675, 440
    d.rectangle([fx0, fy0, fx1, fy1], outline=(*CYAN, 90), width=1)
    d.line((fx0, fy0, fx0 + 130, fy0), fill=(*CYAN2, 190), width=3)

    for x in range(fx0 + 55, fx1, 55):
        d.line((x, fy0, x, fy1), fill=(*CYAN, 12), width=1)
    for y in range(fy0 + 46, fy1, 46):
        d.line((fx0, y, fx1, y), fill=(*CYAN, 12), width=1)

    # Threat sectors
    threat_alpha = int(18 + 70 * alert)
    d.polygon(
        [(520, 130), (660, 92), (675, 210), (575, 238)],
        fill=(*RED, threat_alpha),
    )
    d.polygon(
        [(455, 335), (675, 310), (675, 440), (500, 420)],
        fill=(*YELLOW, int(12 + 35 * alert)),
    )

    d.text((535, 102), "THREAT SECTOR", font=FONT_TINY, fill=(*RED, int(120 + 90 * alert)))

    # Positions
    positions = []
    for drone in SWARM:
        p = drone_pos(drone, phase)
        positions.append((drone, p))

    # Formation links
    for group, color in [(0, CYAN), (1, GREEN), (2, YELLOW)]:
        group_pts = [p for dr, p in positions if dr["group"] == group]
        center = formation_center(group, phase)

        for p in group_pts[::2]:
            d.line([tuple(center), tuple(p)], fill=(*color, 22), width=1)

        d.ellipse([center[0]-7, center[1]-7, center[0]+7, center[1]+7], outline=(*color, 130), width=2)
        d.text((center[0]+10, center[1]-8), f"FORM-{group+1}", font=FONT_TINY, fill=(*color, 150))

    # Drones
    for drone, p in positions:
        x, y = p
        if not (fx0 < x < fx1 and fy0 < y < fy1):
            continue

        group = drone["group"]
        color = [CYAN2, GREEN, YELLOW][group]

        pulse = 0.55 + 0.45 * np.sin(phase * 5 + drone["phase"])
        alpha = int(120 + 100 * pulse)

        d.ellipse([x-3, y-3, x+3, y+3], fill=(*color, alpha))

        if drone["id"] % 9 == 0 or alert > 0.35 and group == 1:
            draw_bracket(d, x, y, 13, color, alpha)
            d.text((x + 15, y - 8), f"D-{drone['id']:02d}", font=FONT_TINY, fill=(*color, 160))

        # velocity hint
        p2 = drone_pos(drone, phase + 0.06)
        v = p2 - p
        n = max(np.hypot(v[0], v[1]), 1)
        v = v / n
        d.line((x, y, x + v[0] * 12, y + v[1] * 12), fill=(*color, 80), width=1)

    # Right panel
    px, py = 710, 92
    pw, ph = 215, 330

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 95, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "SWARM STATE", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("UNITS", f"{N:03d}"),
        ("FORM", "TRIAD"),
        ("SYNC", f"{94 + 4*np.sin(phase):04.1f}%"),
        ("THREAT", "HIGH" if alert > 0.35 else "LOW"),
        ("VECTOR", "SOLVE"),
        ("STATE", "ALERT" if alert > 0.35 else "TRACK"),
    ]

    for m, (k, v) in enumerate(metrics):
        yy = py + 56 + m * 40
        col = RED if v in ("HIGH", "ALERT") else CYAN2
        d.text((px + 16, yy), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 92, yy), v, font=FONT_SMALL, fill=(*col, 220))

    # Formation bars
    bx, by = px + 18, py + ph - 42
    for k in range(24):
        a = int(35 + 100 * np.sin(phase * 4 + k * 0.6) ** 2)
        h = 5 + 20 * np.sin(phase * 1.7 + k * 0.4) ** 2
        d.line((bx + k * 7, by, bx + k * 7, by - h), fill=(*CYAN2, a), width=2)

    # Alert overlay
    if alert > 0.06:
        a = int(150 * alert)
        d.rectangle([fx0, fy0, fx1, fy1], outline=(*RED, a), width=3)
        d.text((fx0 + 18, fy1 - 28), "FORMATION CONFLICT PREDICTED", font=FONT_TINY, fill=(*RED, min(255, a + 40)))

    d.text(
        (32, H - 28),
        "FORMATION SOLVER • MULTI-UNIT VECTOR FIELD • THREAT SECTOR ANALYSIS",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.5))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "drone_swarm_tactical.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/swarm/drone_swarm_tactical.gif


In [15]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/swarm")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 540, 960
FPS = 24
FRAMES = 240

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(1204)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)

N = 54
SWARM = []

for idx in range(N):
    SWARM.append({
        "id": idx,
        "group": idx % 3,
        "phase": rng.uniform(0, 2 * np.pi),
        "radius": rng.uniform(14, 110),
        "angle": rng.uniform(0, 2 * np.pi),
        "speed": rng.uniform(0.35, 1.25),
    })


FIELD = [36, 78, W - 36, 610]


def formation_center(group, phase):
    return [
        np.array([205 + 18 * np.sin(phase * 0.7), 235 + 16 * np.cos(phase * 0.9)]),
        np.array([330 + 22 * np.cos(phase * 0.6), 300 + 20 * np.sin(phase * 0.8)]),
        np.array([285 + 20 * np.sin(phase * 0.5), 450 + 16 * np.cos(phase * 0.7)]),
    ][group]


def drone_pos(drone, phase):
    center = formation_center(drone["group"], phase)
    a = drone["angle"] + phase * drone["speed"]

    if drone["group"] == 0:
        x = center[0] + np.cos(a) * drone["radius"] * 0.72
        y = center[1] + np.sin(a) * drone["radius"] * 0.50
    elif drone["group"] == 1:
        lane = (drone["id"] % 7) - 3
        row = (drone["id"] // 7) % 4
        x = center[0] + lane * 20 + 14 * np.sin(phase * 1.7 + drone["phase"])
        y = center[1] + row * 20 + 18 * np.cos(phase * 1.1 + drone["phase"])
    else:
        x = center[0] + np.cos(a) * drone["radius"] * 0.55
        y = center[1] + np.sin(a * 1.6) * drone["radius"] * 0.42

    return np.array([x, y])


def draw_bracket(d, x, y, s, color, alpha):
    c = s * 0.4
    d.line((x-s, y-s, x-s+c, y-s), fill=(*color, alpha), width=1)
    d.line((x-s, y-s, x-s, y-s+c), fill=(*color, alpha), width=1)
    d.line((x+s, y-s, x+s-c, y-s), fill=(*color, alpha), width=1)
    d.line((x+s, y-s, x+s, y-s+c), fill=(*color, alpha), width=1)
    d.line((x-s, y+s, x-s+c, y+s), fill=(*color, alpha), width=1)
    d.line((x-s, y+s, x-s, y+s-c), fill=(*color, alpha), width=1)
    d.line((x+s, y+s, x+s-c, y+s), fill=(*color, alpha), width=1)
    d.line((x+s, y+s, x+s, y+s-c), fill=(*color, alpha), width=1)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES
    alert = np.exp(-0.5 * ((i - 112) / 11) ** 2)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((28, 12), "DRONE SWARM TACTICAL", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((24, 38, 230, 38), fill=(*CYAN, 170), width=2)
    d.line((230, 38, 254, 52), fill=(*CYAN, 170), width=2)
    d.line((254, 52, W - 28, 52), fill=(*CYAN, 75), width=1)

    fx0, fy0, fx1, fy1 = FIELD

    d.rectangle([fx0, fy0, fx1, fy1], outline=(*CYAN, 90), width=1)
    d.line((fx0, fy0, fx0 + 130, fy0), fill=(*CYAN2, 190), width=3)

    for x in range(fx0 + 40, fx1, 40):
        d.line((x, fy0, x, fy1), fill=(*CYAN, 12), width=1)

    for y in range(fy0 + 45, fy1, 45):
        d.line((fx0, y, fx1, y), fill=(*CYAN, 12), width=1)

    threat_alpha = int(18 + 70 * alert)

    d.polygon(
        [(345, 145), (505, 110), (504, 260), (382, 275)],
        fill=(*RED, threat_alpha),
    )

    d.polygon(
        [(310, 430), (505, 405), (504, 610), (330, 585)],
        fill=(*YELLOW, int(12 + 35 * alert)),
    )

    d.text((350, 120), "THREAT SECTOR", font=FONT_TINY, fill=(*RED, int(120 + 90 * alert)))

    positions = [(drone, drone_pos(drone, phase)) for drone in SWARM]

    for group, color in [(0, CYAN), (1, GREEN), (2, YELLOW)]:
        group_pts = [p for dr, p in positions if dr["group"] == group]
        center = formation_center(group, phase)

        for p in group_pts[::2]:
            if fx0 < p[0] < fx1 and fy0 < p[1] < fy1:
                d.line([tuple(center), tuple(p)], fill=(*color, 22), width=1)

        d.ellipse([center[0]-7, center[1]-7, center[0]+7, center[1]+7], outline=(*color, 130), width=2)
        d.text((center[0]+10, center[1]-8), f"FORM-{group+1}", font=FONT_TINY, fill=(*color, 150))

    for drone, p in positions:
        x, y = p

        if not (fx0 < x < fx1 and fy0 < y < fy1):
            continue

        group = drone["group"]
        color = [CYAN2, GREEN, YELLOW][group]

        pulse = 0.55 + 0.45 * np.sin(phase * 5 + drone["phase"])
        alpha = int(120 + 100 * pulse)

        d.ellipse([x-3, y-3, x+3, y+3], fill=(*color, alpha))

        if drone["id"] % 9 == 0 or (alert > 0.35 and group == 1):
            draw_bracket(d, x, y, 13, color, alpha)
            d.text((x + 15, y - 8), f"D-{drone['id']:02d}", font=FONT_TINY, fill=(*color, 160))

        p2 = drone_pos(drone, phase + 0.06)
        v = p2 - p
        n = max(np.hypot(v[0], v[1]), 1)
        v = v / n
        d.line((x, y, x + v[0] * 12, y + v[1] * 12), fill=(*color, 80), width=1)

    # bottom panel
    px, py = 36, 640
    pw, ph = W - 72, 250

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 120, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "SWARM STATE", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("UNITS", f"{N:03d}"),
        ("FORM", "TRIAD"),
        ("SYNC", f"{94 + 4*np.sin(phase):04.1f}%"),
        ("THREAT", "HIGH" if alert > 0.35 else "LOW"),
        ("VECTOR", "SOLVE"),
        ("STATE", "ALERT" if alert > 0.35 else "TRACK"),
    ]

    for m, (k, v) in enumerate(metrics):
        yy = py + 56 + m * 28
        col = RED if v in ("HIGH", "ALERT") else CYAN2

        d.text((px + 18, yy), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 112, yy), v, font=FONT_SMALL, fill=(*col, 220))

        for t in range(10):
            a = int(35 + 90 * np.sin(phase * 4 + m + t) ** 2)
            tx = px + 285 + t * 13
            d.line((tx, yy + 8, tx + 8, yy + 8), fill=(*CYAN, a), width=1)

    bx, by = px + 18, py + ph - 36

    for k in range(32):
        a = int(35 + 100 * np.sin(phase * 4 + k * 0.6) ** 2)
        h = 5 + 20 * np.sin(phase * 1.7 + k * 0.4) ** 2
        d.line((bx + k * 12, by, bx + k * 12, by - h), fill=(*CYAN2, a), width=2)

    if alert > 0.06:
        a = int(150 * alert)
        d.rectangle([fx0, fy0, fx1, fy1], outline=(*RED, a), width=3)
        d.text((fx0 + 18, fy1 - 28), "FORMATION CONFLICT PREDICTED", font=FONT_TINY, fill=(*RED, min(255, a + 40)))

    d.text(
        (28, H - 34),
        "FORMATION SOLVER • MULTI-UNIT VECTOR FIELD • THREAT SECTOR ANALYSIS",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 40, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 40, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.5))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "drone_swarm_tactical_vertical.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/swarm/drone_swarm_tactical_vertical.gif


In [16]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/memory")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(2605)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


MEM_BLOCKS = []
cols, rows = 4, 3
x0, y0 = 60, 92
bw, bh = 130, 82
gap_x, gap_y = 18, 18

for r in range(rows):
    for c in range(cols):
        MEM_BLOCKS.append({
            "x": x0 + c * (bw + gap_x),
            "y": y0 + r * (bh + gap_y),
            "delay": 12 + (r * cols + c) * 7 + rng.uniform(-3, 3),
            "damage": rng.random() < 0.28,
            "seed": rng.integers(0, 9999),
        })


def draw_memory_tile(d, x, y, w, h, phase, alpha, damaged, seed):
    local_rng = np.random.default_rng(seed)

    d.rectangle([x, y, x + w, y + h], outline=(*CYAN, int(95 * alpha)), width=1)
    d.rectangle([x + 2, y + 2, x + w - 2, y + h - 2], fill=(*CYAN, int(14 * alpha)))

    # abstract memory image: face-like / scene-like fragments
    cx = x + w / 2
    cy = y + h / 2

    if damaged:
        col = RED
    else:
        col = CYAN2

    for k in range(7):
        px = x + 12 + local_rng.random() * (w - 24)
        py = y + 10 + local_rng.random() * (h - 20)
        rr = 5 + local_rng.random() * 14
        a = int((28 + 55 * np.sin(phase * 2 + k + seed) ** 2) * alpha)
        d.ellipse([px - rr, py - rr, px + rr, py + rr], outline=(*col, a), width=1)

    # synthetic silhouette hint
    d.ellipse([cx - 18, cy - 28, cx + 18, cy + 8], outline=(*col, int(95 * alpha)), width=1)
    d.line([cx, cy + 8, cx, cy + 32], fill=(*col, int(80 * alpha)), width=1)
    d.arc([cx - 32, cy - 6, cx + 32, cy + 54], 205, 335, fill=(*col, int(60 * alpha)), width=1)

    # scanlines
    for yy in range(int(y + 8), int(y + h - 6), 9):
        a = int((20 + 30 * np.sin(phase * 4 + yy * 0.1) ** 2) * alpha)
        d.line([x + 8, yy, x + w - 8, yy], fill=(*CYAN, a), width=1)

    if damaged:
        for _ in range(4):
            gy = y + local_rng.integers(8, int(h - 8))
            gx = x + local_rng.integers(4, int(w - 40))
            d.rectangle([gx, gy, gx + local_rng.integers(22, 68), gy + 4], fill=(*RED, int(80 * alpha)))


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES
    reconstruction = smoothstep((i - 30) / 115)
    lock = smoothstep((i - 128) / 35)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((32, 10), "SYNTHETIC MEMORY LOADER", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 245, 34), fill=(*CYAN, 170), width=2)
    d.line((245, 34, 268, 48), fill=(*CYAN, 170), width=2)
    d.line((268, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Memory matrix frame
    mx0, my0, mx1, my1 = 42, 72, 650, 395
    d.rectangle([mx0, my0, mx1, my1], outline=(*CYAN, 85), width=1)
    d.line((mx0, my0, mx0 + 145, my0), fill=(*CYAN2, 190), width=3)

    restored = 0
    damaged = 0

    for idx, block in enumerate(MEM_BLOCKS):
        a = smoothstep((i - block["delay"]) / 18)

        if a <= 0:
            continue

        is_damaged = block["damage"] and i < 145

        if a > 0.85 and not is_damaged:
            restored += 1

        if is_damaged:
            damaged += 1

        draw_memory_tile(
            d,
            block["x"],
            block["y"],
            bw,
            bh,
            phase,
            a,
            is_damaged,
            block["seed"],
        )

    # Neural reconstruction web
    if reconstruction > 0.1:
        center = np.array([350, 245])
        nodes = []
        for k in range(16):
            ang = phase * 0.18 + k * 2 * np.pi / 16
            rr = 80 + 55 * np.sin(phase * 0.7 + k) ** 2
            nodes.append(center + np.array([np.cos(ang) * rr, np.sin(ang) * rr * 0.62]))

        for a_idx in range(len(nodes)):
            for b_idx in range(a_idx + 1, len(nodes)):
                if (a_idx + b_idx) % 5 == 0:
                    alpha = int(28 * reconstruction)
                    d.line([tuple(nodes[a_idx]), tuple(nodes[b_idx])], fill=(*CYAN, alpha), width=1)

        for k, node in enumerate(nodes):
            pulse = 0.55 + 0.45 * np.sin(phase * 4 + k)
            d.ellipse([node[0] - 3, node[1] - 3, node[0] + 3, node[1] + 3], fill=(*GREEN, int(150 * reconstruction * pulse)))

    # Right diagnostics panel
    px, py = 690, 92
    pw, ph = 230, 330

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 100, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "MEMORY INDEX", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("FRAMES", f"{restored:02d}/{len(MEM_BLOCKS):02d}"),
        ("DAMAGE", f"{damaged:02d}"),
        ("SYNC", f"{reconstruction*100:04.1f}%"),
        ("IDENT", "ZANE-UNIT"),
        ("LOAD", f"{lock*100:04.1f}%"),
        ("STATE", "RESTORED" if lock > 0.85 else "REBUILD"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 40
        col = GREEN if v in ("RESTORED", "ZANE-UNIT") else CYAN2
        if k == "DAMAGE" and damaged > 0:
            col = YELLOW

        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 94, y), v, font=FONT_SMALL, fill=(*col, 220))

    # Event log
    log_x, log_y = 55, 420
    logs = [
        "MEMORY SECTOR 03: FRAGMENTED",
        "SYNTHETIC RECALL STREAM: ACTIVE",
        "IDENTITY ANCHOR: PARTIAL",
        "NEURAL MAP: RECONSTRUCTING",
    ]

    for k, line in enumerate(logs):
        a = int(80 + 80 * np.sin(phase * 3 + k) ** 2)
        d.text((log_x, log_y + k * 22), line, font=FONT_TINY, fill=(*CYAN, a))

    # Loading bar
    bx, by = px + 18, py + ph - 36
    bw_bar, bh_bar = pw - 36, 12

    d.rectangle([bx, by, bx + bw_bar, by + bh_bar], outline=(*CYAN, 90), width=1)
    d.rectangle([bx + 2, by + 2, bx + 2 + int((bw_bar - 4) * lock), by + bh_bar - 2], fill=(*GREEN, 170))

    # Glitch pulse
    glitch = np.exp(-0.5 * ((i - 88) / 7) ** 2)
    if glitch > 0.05:
        a = int(140 * glitch)
        d.rectangle([mx0, my0, mx1, my1], outline=(*RED, a), width=3)
        for _ in range(10):
            gx = int(rng.integers(mx0 + 5, mx1 - 80))
            gy = int(rng.integers(my0 + 5, my1 - 5))
            d.rectangle([gx, gy, gx + int(rng.integers(40, 150)), gy + 3], fill=(*RED, int(rng.integers(30, 90) * glitch)))

    d.text(
        (32, H - 28),
        "FRAGMENTED MEMORY • SYNTHETIC RECALL • IDENTITY ANCHOR RECONSTRUCTION",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.5))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "synthetic_memory_loader.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/memory/synthetic_memory_loader.gif


In [17]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/cartography")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)

rng = np.random.default_rng(3301)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)

STARS = []
for _ in range(260):
    STARS.append({
        "x": rng.uniform(70, 650),
        "y": rng.uniform(85, 430),
        "mag": rng.uniform(0.4, 1.0),
        "phase": rng.uniform(0, 2 * np.pi),
    })

BEACONS = [
    ("PSR-014", 180, 150, CYAN2),
    ("GATE-7", 420, 235, GREEN),
    ("RELAY", 560, 135, YELLOW),
    ("VOID-MARK", 330, 365, PURPLE),
]

ROUTE = np.array([
    [120, 390],
    [210, 330],
    [320, 290],
    [420, 235],
    [545, 188],
    [610, 115],
])


def draw_nebula(d, cx, cy, rx, ry, phase, color, alpha):
    pts = []
    for k in range(96):
        a = k * 2 * np.pi / 96
        wobble = 1 + 0.12 * np.sin(a * 5 + phase) + 0.08 * np.cos(a * 3 - phase * 0.7)
        x = cx + np.cos(a) * rx * wobble
        y = cy + np.sin(a) * ry * wobble
        pts.append((x, y))

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*color, alpha), width=1)
    d.line([pts[-1], pts[0]], fill=(*color, alpha), width=1)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((32, 10), "STELLAR CARTOGRAPHY", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 225, 34), fill=(*CYAN, 170), width=2)
    d.line((225, 34, 248, 48), fill=(*CYAN, 170), width=2)
    d.line((248, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Map frame
    mx0, my0, mx1, my1 = 45, 72, 675, 435
    d.rectangle([mx0, my0, mx1, my1], outline=(*CYAN, 90), width=1)
    d.line((mx0, my0, mx0 + 150, my0), fill=(*CYAN2, 190), width=3)

    # Grid
    for x in range(mx0 + 50, mx1, 50):
        d.line((x, my0, x, my1), fill=(*CYAN, 12), width=1)
    for y in range(my0 + 45, my1, 45):
        d.line((mx0, y, mx1, y), fill=(*CYAN, 12), width=1)

    # Nebula contours
    draw_nebula(d, 245, 210, 130, 58, phase * 0.8, PURPLE, 45)
    draw_nebula(d, 510, 310, 115, 72, phase * 0.6 + 2, CYAN, 38)
    draw_nebula(d, 405, 150, 82, 42, phase * 0.9 + 1, GREEN, 34)

    # Stars
    for s in STARS:
        pulse = 0.65 + 0.35 * np.sin(phase * 2 + s["phase"])
        a = int((70 + 150 * s["mag"]) * pulse)
        r = 1 + 1.8 * s["mag"]
        d.ellipse([s["x"] - r, s["y"] - r, s["x"] + r, s["y"] + r], fill=(*WHITE, a))

    # Route
    for idx, (p1, p2) in enumerate(zip(ROUTE[:-1], ROUTE[1:])):
        a = int(100 + 70 * np.sin(phase * 2 + idx) ** 2)
        d.line([tuple(p1), tuple(p2)], fill=(*CYAN2, a), width=2)

    route_idx = int((i * 2) % len(ROUTE))
    for k, p in enumerate(ROUTE):
        col = GREEN if k == route_idx % len(ROUTE) else CYAN2
        d.ellipse([p[0]-5, p[1]-5, p[0]+5, p[1]+5], fill=(*col, 190))

    # Beacons
    for idx, (name, x, y, col) in enumerate(BEACONS):
        pulse = 0.55 + 0.45 * np.sin(phase * 4 + idx)
        d.ellipse([x - 16, y - 16, x + 16, y + 16], outline=(*col, int(120 * pulse)), width=2)
        d.ellipse([x - 4, y - 4, x + 4, y + 4], fill=(*col, 220))
        d.text((x + 18, y - 8), name, font=FONT_TINY, fill=(*col, 165))

    # Scanning beam
    scan_x = mx0 + int((i * 4) % (mx1 - mx0))
    for dx in range(-8, 9):
        a = max(0, 45 - abs(dx) * 5)
        d.line((scan_x + dx, my0 + 2, scan_x + dx, my1 - 2), fill=(*CYAN2, a), width=1)

    # Right panel
    px, py = 710, 92
    pw, ph = 215, 330

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 95, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "SECTOR INDEX", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("SECTOR", "LUM-32"),
        ("STARS", f"{len(STARS):03d}"),
        ("BEACONS", f"{len(BEACONS):02d}"),
        ("ROUTE", "ACTIVE"),
        ("DENSITY", f"{72 + 8*np.sin(phase):04.1f}%"),
        ("STATE", "MAPPED"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 40
        col = GREEN if v in ("ACTIVE", "MAPPED") else CYAN2
        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 94, y), v, font=FONT_SMALL, fill=(*col, 220))

    d.text(
        (32, H - 28),
        "SECTOR MAP • BEACON ROUTES • NEBULA CONTOURS • STELLAR DENSITY SCAN",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.4))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "stellar_cartography.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/cartography/stellar_cartography.gif


In [18]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/cartography")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 540, 960
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)

rng = np.random.default_rng(3301)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


MAP_X0, MAP_Y0 = 36, 78
MAP_X1, MAP_Y1 = W - 36, 610

STARS = []
for _ in range(260):
    STARS.append({
        "x": rng.uniform(MAP_X0 + 25, MAP_X1 - 25),
        "y": rng.uniform(MAP_Y0 + 25, MAP_Y1 - 25),
        "mag": rng.uniform(0.4, 1.0),
        "phase": rng.uniform(0, 2 * np.pi),
    })

BEACONS = [
    ("PSR-014", 145, 170, CYAN2),
    ("GATE-7", 295, 305, GREEN),
    ("RELAY", 405, 205, YELLOW),
    ("VOID-MARK", 245, 480, PURPLE),
]

ROUTE = np.array([
    [100, 535],
    [165, 455],
    [230, 390],
    [295, 305],
    [365, 245],
    [420, 165],
])


def draw_nebula(d, cx, cy, rx, ry, phase, color, alpha):
    pts = []
    for k in range(96):
        a = k * 2 * np.pi / 96
        wobble = 1 + 0.12 * np.sin(a * 5 + phase) + 0.08 * np.cos(a * 3 - phase * 0.7)
        pts.append((cx + np.cos(a) * rx * wobble, cy + np.sin(a) * ry * wobble))

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*color, alpha), width=1)
    d.line([pts[-1], pts[0]], fill=(*color, alpha), width=1)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((28, 12), "STELLAR CARTOGRAPHY", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((24, 38, 225, 38), fill=(*CYAN, 170), width=2)
    d.line((225, 38, 249, 52), fill=(*CYAN, 170), width=2)
    d.line((249, 52, W - 28, 52), fill=(*CYAN, 75), width=1)

    d.rectangle([MAP_X0, MAP_Y0, MAP_X1, MAP_Y1], outline=(*CYAN, 90), width=1)
    d.line((MAP_X0, MAP_Y0, MAP_X0 + 140, MAP_Y0), fill=(*CYAN2, 190), width=3)

    for x in range(MAP_X0 + 40, MAP_X1, 40):
        d.line((x, MAP_Y0, x, MAP_Y1), fill=(*CYAN, 12), width=1)
    for y in range(MAP_Y0 + 45, MAP_Y1, 45):
        d.line((MAP_X0, y, MAP_X1, y), fill=(*CYAN, 12), width=1)

    draw_nebula(d, 195, 250, 95, 65, phase * 0.8, PURPLE, 45)
    draw_nebula(d, 360, 430, 88, 78, phase * 0.6 + 2, CYAN, 38)
    draw_nebula(d, 325, 165, 70, 42, phase * 0.9 + 1, GREEN, 34)

    for s in STARS:
        pulse = 0.65 + 0.35 * np.sin(phase * 2 + s["phase"])
        a = int((70 + 150 * s["mag"]) * pulse)
        r = 1 + 1.8 * s["mag"]
        d.ellipse([s["x"] - r, s["y"] - r, s["x"] + r, s["y"] + r], fill=(*WHITE, a))

    for idx, (p1, p2) in enumerate(zip(ROUTE[:-1], ROUTE[1:])):
        a = int(100 + 70 * np.sin(phase * 2 + idx) ** 2)
        d.line([tuple(p1), tuple(p2)], fill=(*CYAN2, a), width=2)

    route_idx = int((i * 2) % len(ROUTE))
    for k, p in enumerate(ROUTE):
        col = GREEN if k == route_idx % len(ROUTE) else CYAN2
        d.ellipse([p[0] - 5, p[1] - 5, p[0] + 5, p[1] + 5], fill=(*col, 190))

    for idx, (name, x, y, col) in enumerate(BEACONS):
        pulse = 0.55 + 0.45 * np.sin(phase * 4 + idx)
        d.ellipse([x - 16, y - 16, x + 16, y + 16], outline=(*col, int(120 * pulse)), width=2)
        d.ellipse([x - 4, y - 4, x + 4, y + 4], fill=(*col, 220))
        d.text((x + 18, y - 8), name, font=FONT_TINY, fill=(*col, 165))

    scan_x = MAP_X0 + int((i * 3.2) % (MAP_X1 - MAP_X0))
    for dx in range(-8, 9):
        a = max(0, 45 - abs(dx) * 5)
        d.line((scan_x + dx, MAP_Y0 + 2, scan_x + dx, MAP_Y1 - 2), fill=(*CYAN2, a), width=1)

    # Bottom sector panel
    px, py = 36, 640
    pw, ph = W - 72, 250

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 125, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "SECTOR INDEX", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("SECTOR", "LUM-32"),
        ("STARS", f"{len(STARS):03d}"),
        ("BEACONS", f"{len(BEACONS):02d}"),
        ("ROUTE", "ACTIVE"),
        ("DENSITY", f"{72 + 8*np.sin(phase):04.1f}%"),
        ("STATE", "MAPPED"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 28
        col = GREEN if v in ("ACTIVE", "MAPPED") else CYAN2

        d.text((px + 18, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 120, y), v, font=FONT_SMALL, fill=(*col, 220))

        for t in range(10):
            a = int(35 + 90 * np.sin(phase * 4 + m + t) ** 2)
            tx = px + 285 + t * 13
            d.line((tx, y + 8, tx + 8, y + 8), fill=(*CYAN, a), width=1)

    bx, by = px + 18, py + ph - 36
    for k in range(32):
        a = int(35 + 100 * np.sin(phase * 4 + k * 0.6) ** 2)
        h = 5 + 20 * np.sin(phase * 1.7 + k * 0.4) ** 2
        d.line((bx + k * 12, by, bx + k * 12, by - h), fill=(*CYAN2, a), width=2)

    d.text(
        (28, H - 34),
        "SECTOR MAP • BEACON ROUTES • NEBULA CONTOURS",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 40, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 40, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.4))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "stellar_cartography_vertical.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/cartography/stellar_cartography_vertical.gif


In [19]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/quantum")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
RED = (255, 80, 110)
PURPLE = (190, 120, 255)
WHITE = (245, 250, 255)

rng = np.random.default_rng(8801)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def draw_probability_cloud(d, cx, cy, phase, color, strength):
    for k in range(42):
        ang = k * 2 * np.pi / 42 + 0.2 * np.sin(phase + k)
        rr = 20 + 95 * (0.45 + 0.55 * np.sin(phase * 0.7 + k * 0.9) ** 2)
        x = cx + np.cos(ang) * rr
        y = cy + np.sin(ang) * rr * 0.72

        a = int((18 + 70 * np.sin(phase * 3 + k) ** 2) * strength)
        r = 2 + 3 * np.sin(phase * 2 + k) ** 2

        d.ellipse([x-r, y-r, x+r, y+r], fill=(*color, a))


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    lock = smoothstep((i - 35) / 65)
    collapse = np.exp(-0.5 * ((i - 132) / 9) ** 2)
    decoherence = np.exp(-0.5 * ((i - 92) / 12) ** 2)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((32, 10), "QUANTUM ENTANGLEMENT MONITOR", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 290, 34), fill=(*CYAN, 170), width=2)
    d.line((290, 34, 314, 48), fill=(*CYAN, 170), width=2)
    d.line((314, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Main frame
    fx0, fy0, fx1, fy1 = 50, 72, 665, 425
    d.rectangle([fx0, fy0, fx1, fy1], outline=(*CYAN, 90), width=1)
    d.line((fx0, fy0, fx0 + 145, fy0), fill=(*CYAN2, 190), width=3)

    # Grid
    for x in range(fx0 + 50, fx1, 50):
        d.line((x, fy0, x, fy1), fill=(*CYAN, 12), width=1)
    for y in range(fy0 + 45, fy1, 45):
        d.line((fx0, y, fx1, y), fill=(*CYAN, 12), width=1)

    p1 = np.array([220 + 18 * np.sin(phase * 0.6), 250 + 12 * np.cos(phase * 0.8)])
    p2 = np.array([500 + 20 * np.cos(phase * 0.5), 250 + 14 * np.sin(phase * 0.7)])

    draw_probability_cloud(d, p1[0], p1[1], phase, CYAN2, 0.9)
    draw_probability_cloud(d, p2[0], p2[1], phase + np.pi, PURPLE, 0.9)

    # Entanglement bridge
    for k in range(5):
        offset = (k - 2) * 9
        pts = []
        for t in np.linspace(0, 1, 90):
            base = p1 * (1 - t) + p2 * t
            wave = np.array([0, np.sin(t * np.pi * 4 + phase * 2 + k) * (18 + offset)])
            pts.append(base + wave)

        a = int((25 + 55 * np.sin(phase * 3 + k) ** 2) * lock)
        for a1, a2 in zip(pts[:-1], pts[1:]):
            d.line([tuple(a1), tuple(a2)], fill=(*CYAN, a), width=1)

    # Particle nodes
    for idx, (p, col, label) in enumerate([(p1, CYAN2, "Q-A"), (p2, PURPLE, "Q-B")]):
        pulse = 0.55 + 0.45 * np.sin(phase * 5 + idx * np.pi)
        r = 16 + 8 * pulse

        d.ellipse([p[0]-r, p[1]-r, p[0]+r, p[1]+r], outline=(*col, int(160 * pulse)), width=2)
        d.ellipse([p[0]-5, p[1]-5, p[0]+5, p[1]+5], fill=(*WHITE, 220))
        d.text((p[0] + 24, p[1] - 8), label, font=FONT_TINY, fill=(*col, 180))

    # Decoherence field
    if decoherence > 0.05:
        a = int(120 * decoherence)
        for _ in range(14):
            y = int(rng.integers(fy0 + 10, fy1 - 10))
            x = int(rng.integers(fx0 + 10, fx1 - 120))
            d.rectangle([x, y, x + int(rng.integers(40, 160)), y + 3], fill=(*RED, a))
        d.text((fx0 + 18, fy1 - 30), "DECOHERENCE SPIKE", font=FONT_TINY, fill=(*RED, min(255, a + 50)))

    # Collapse flash
    if collapse > 0.04:
        a = int(150 * collapse)
        d.line([tuple(p1), tuple(p2)], fill=(*YELLOW, a), width=4)
        d.rectangle([fx0, fy0, fx1, fy1], outline=(*YELLOW, a), width=3)

    # Right panel
    px, py = 705, 92
    pw, ph = 220, 330

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 105, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "PAIR STATE", font=FONT_SMALL, fill=(*CYAN2, 190))

    coherence = 34 + 58 * lock - 24 * decoherence + 16 * collapse
    entropy = 77 - 45 * lock + 30 * decoherence
    bell = 2.1 + 0.38 * lock + 0.2 * np.sin(phase * 1.3)

    metrics = [
        ("PAIR", "Q-A / Q-B"),
        ("COHER", f"{coherence:04.1f}%"),
        ("ENTROPY", f"{entropy:04.1f}%"),
        ("BELL", f"{bell:.2f}"),
        ("PHASE", f"{lock*100:04.1f}%"),
        ("STATE", "COLLAPSE" if collapse > 0.3 else "LOCK"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 40
        col = YELLOW if v == "COLLAPSE" else CYAN2
        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 96, y), v, font=FONT_SMALL, fill=(*col, 220))

    # Mini phase bars
    bx, by = px + 18, py + ph - 36
    for k in range(24):
        a = int(35 + 100 * np.sin(phase * 4 + k * 0.6) ** 2)
        h = 5 + 20 * np.sin(phase * 1.7 + k * 0.4) ** 2
        d.line((bx + k * 7, by, bx + k * 7, by - h), fill=(*CYAN2, a), width=2)

    d.text(
        (32, H - 28),
        "ENTANGLED PAIR LOCK • PROBABILITY CLOUD • DECOHERENCE MONITOR • COLLAPSE EVENT",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.4))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "quantum_entanglement_monitor.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/quantum/quantum_entanglement_monitor.gif


In [20]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/descent")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(4412)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES
    descent = smoothstep((i - 12) / 135)
    lock = smoothstep((i - 120) / 38)
    heating = np.exp(-0.5 * ((i - 72) / 20) ** 2)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((32, 10), "PLANETARY DESCENT HUD", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 230, 34), fill=(*CYAN, 170), width=2)
    d.line((230, 34, 254, 48), fill=(*CYAN, 170), width=2)
    d.line((254, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Main descent frame
    fx0, fy0, fx1, fy1 = 58, 72, 660, 430
    d.rectangle([fx0, fy0, fx1, fy1], outline=(*CYAN, 90), width=1)
    d.line((fx0, fy0, fx0 + 145, fy0), fill=(*CYAN2, 190), width=3)

    cx = (fx0 + fx1) / 2
    horizon_y = fy0 + 205 + 22 * np.sin(phase * 0.8)

    # Sky grid
    for x in range(fx0 + 50, fx1, 50):
        d.line((x, fy0, x, fy1), fill=(*CYAN, 10), width=1)
    for y in range(fy0 + 45, fy1, 45):
        d.line((fx0, y, fx1, y), fill=(*CYAN, 10), width=1)

    # Terrain wireframe
    terrain_base = fy1 - 34
    xs = np.linspace(fx0 + 15, fx1 - 15, 38)
    heights = 38 + 24 * np.sin(xs * 0.035 + phase * 0.5) + 16 * np.sin(xs * 0.087 - phase)
    pts = [(x, terrain_base - h) for x, h in zip(xs, heights)]

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*CYAN2, 105), width=1)
        d.line([p1, (p1[0], terrain_base)], fill=(*CYAN, 22), width=1)

    # Perspective terrain depth lines
    for k in range(8):
        y = terrain_base - k * 28
        shrink = k * 24
        d.line(
            (fx0 + 20 + shrink, y, fx1 - 20 - shrink, y),
            fill=(*CYAN, max(8, 40 - k * 4)),
            width=1,
        )

    # Landing ellipse
    lx = cx + 52 * np.sin(phase * 0.6)
    ly = terrain_base - 55
    ell_w = 118 - 38 * lock
    ell_h = 36 - 12 * lock

    d.ellipse(
        [lx - ell_w, ly - ell_h, lx + ell_w, ly + ell_h],
        outline=(*GREEN, int(90 + 110 * lock)),
        width=2,
    )
    d.text((lx + 76, ly - 8), "LANDING ELLIPSE", font=FONT_TINY, fill=(*GREEN, 165))

    # Hazards
    hazards = [
        (fx0 + 145, terrain_base - 62, "ROCK"),
        (fx0 + 455, terrain_base - 74, "SLOPE"),
        (fx0 + 520, terrain_base - 38, "HEAT"),
    ]

    for idx, (hx, hy, label) in enumerate(hazards):
        pulse = 0.55 + 0.45 * np.sin(phase * 4 + idx)
        col = RED if idx == 2 and heating > 0.25 else YELLOW

        d.polygon(
            [(hx, hy - 12), (hx - 12, hy + 10), (hx + 12, hy + 10)],
            outline=(*col, int(150 * pulse)),
        )
        d.text((hx + 14, hy - 8), label, font=FONT_TINY, fill=(*col, int(165 * pulse)))

    # Descent vector / craft
    craft_x = cx + 48 * np.sin(phase * 0.9) * (1 - 0.45 * lock)
    craft_y = fy0 + 78 + descent * 165

    d.line((craft_x, craft_y, lx, ly), fill=(*GREEN, 90), width=1)

    # Craft marker
    b = 22 + 5 * np.sin(phase * 5)
    d.line((craft_x - b, craft_y - b, craft_x - b + 12, craft_y - b), fill=(*CYAN2, 220), width=2)
    d.line((craft_x - b, craft_y - b, craft_x - b, craft_y - b + 12), fill=(*CYAN2, 220), width=2)
    d.line((craft_x + b, craft_y - b, craft_x + b - 12, craft_y - b), fill=(*CYAN2, 220), width=2)
    d.line((craft_x + b, craft_y - b, craft_x + b, craft_y - b + 12), fill=(*CYAN2, 220), width=2)
    d.line((craft_x - b, craft_y + b, craft_x - b + 12, craft_y + b), fill=(*CYAN2, 220), width=2)
    d.line((craft_x - b, craft_y + b, craft_x - b, craft_y + b - 12), fill=(*CYAN2, 220), width=2)
    d.line((craft_x + b, craft_y + b, craft_x + b - 12, craft_y + b), fill=(*CYAN2, 220), width=2)
    d.line((craft_x + b, craft_y + b, craft_x + b, craft_y + b - 12), fill=(*CYAN2, 220), width=2)

    d.ellipse([craft_x - 4, craft_y - 4, craft_x + 4, craft_y + 4], fill=(*WHITE, 230))

    # Altitude ladder left
    ladder_x = fx0 + 28
    for k in range(12):
        y = fy0 + 42 + k * 27
        tick = 24 if k % 2 == 0 else 14
        d.line((ladder_x, y, ladder_x + tick, y), fill=(*CYAN, 120), width=1)
        if k % 2 == 0:
            alt = int(12_000 * (1 - descent) - k * 450)
            d.text((ladder_x + 30, y - 7), f"{max(0, alt):05d}", font=FONT_TINY, fill=(*CYAN, 120))

    # Velocity vector right
    vbar_x = fx1 - 44
    d.line((vbar_x, fy0 + 42, vbar_x, fy1 - 40), fill=(*CYAN, 65), width=1)
    vy = fy0 + 42 + (fy1 - fy0 - 82) * descent
    d.polygon([(vbar_x, vy), (vbar_x - 8, vy - 10), (vbar_x + 8, vy - 10)], fill=(*GREEN, 180))

    # Heating overlay
    if heating > 0.05:
        a = int(130 * heating)
        d.rectangle([fx0, fy0, fx1, fy1], outline=(*ORANGE, a), width=3)
        d.text((fx0 + 18, fy0 + 330), "ATMOSPHERIC HEATING", font=FONT_TINY, fill=(*ORANGE, min(255, a + 40)))

    # Right telemetry panel
    px, py = 700, 92
    pw, ph = 225, 330

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 105, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "DESCENT SOLVER", font=FONT_SMALL, fill=(*CYAN2, 190))

    alt = 12000 * (1 - descent)
    vel = 3.8 - 2.4 * lock + 0.2 * np.sin(phase)
    heat = 18 + 72 * heating
    slope = 4.2 + 1.5 * np.sin(phase * 1.2)

    metrics = [
        ("ALT", f"{alt:06.0f} m"),
        ("VEL", f"{vel:04.2f} km/s"),
        ("HEAT", f"{heat:04.1f}%"),
        ("SLOPE", f"{slope:03.1f}°"),
        ("LOCK", f"{lock*100:04.1f}%"),
        ("STATE", "LOCK" if lock > 0.85 else "GUIDE"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 40
        col = GREEN if v == "LOCK" else CYAN2
        if k == "HEAT" and heat > 60:
            col = ORANGE

        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 90, y), v, font=FONT_SMALL, fill=(*col, 220))

    # Small attitude bars
    bx, by = px + 18, py + ph - 36
    for k in range(24):
        a = int(35 + 100 * np.sin(phase * 4 + k * 0.6) ** 2)
        h = 5 + 22 * np.sin(phase * 1.7 + k * 0.4) ** 2
        d.line((bx + k * 7, by, bx + k * 7, by - h), fill=(*CYAN2, a), width=2)

    d.text(
        (32, H - 28),
        "ALTITUDE LADDER • TERRAIN WIREFRAME • LANDING ELLIPSE • HAZARD MAP",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.4))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "planetary_descent_hud.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/descent/planetary_descent_hud.gif


In [22]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/descent")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 540, 960
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(4412)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def draw_brackets(d, x, y, b, color, alpha=220):
    c = 12
    d.line((x-b, y-b, x-b+c, y-b), fill=(*color, alpha), width=2)
    d.line((x-b, y-b, x-b, y-b+c), fill=(*color, alpha), width=2)
    d.line((x+b, y-b, x+b-c, y-b), fill=(*color, alpha), width=2)
    d.line((x+b, y-b, x+b, y-b+c), fill=(*color, alpha), width=2)
    d.line((x-b, y+b, x-b+c, y+b), fill=(*color, alpha), width=2)
    d.line((x-b, y+b, x-b, y+b-c), fill=(*color, alpha), width=2)
    d.line((x+b, y+b, x+b-c, y+b), fill=(*color, alpha), width=2)
    d.line((x+b, y+b, x+b, y+b-c), fill=(*color, alpha), width=2)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES
    descent = smoothstep((i - 12) / 135)
    lock = smoothstep((i - 120) / 38)
    heating = np.exp(-0.5 * ((i - 72) / 20) ** 2)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((28, 12), "PLANETARY DESCENT HUD", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((24, 38, 230, 38), fill=(*CYAN, 170), width=2)
    d.line((230, 38, 254, 52), fill=(*CYAN, 170), width=2)
    d.line((254, 52, W - 28, 52), fill=(*CYAN, 75), width=1)

    # Main descent frame
    fx0, fy0, fx1, fy1 = 36, 78, W - 36, 610
    d.rectangle([fx0, fy0, fx1, fy1], outline=(*CYAN, 90), width=1)
    d.line((fx0, fy0, fx0 + 145, fy0), fill=(*CYAN2, 190), width=3)

    cx = (fx0 + fx1) / 2

    # Grid
    for x in range(fx0 + 40, fx1, 40):
        d.line((x, fy0, x, fy1), fill=(*CYAN, 10), width=1)

    for y in range(fy0 + 45, fy1, 45):
        d.line((fx0, y, fx1, y), fill=(*CYAN, 10), width=1)

    # Terrain wireframe
    terrain_base = fy1 - 42
    xs = np.linspace(fx0 + 14, fx1 - 14, 32)
    heights = 42 + 26 * np.sin(xs * 0.035 + phase * 0.5) + 15 * np.sin(xs * 0.087 - phase)
    pts = [(x, terrain_base - h) for x, h in zip(xs, heights)]

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*CYAN2, 105), width=1)
        d.line([p1, (p1[0], terrain_base)], fill=(*CYAN, 22), width=1)

    for k in range(9):
        y = terrain_base - k * 34
        shrink = k * 18
        d.line(
            (fx0 + 18 + shrink, y, fx1 - 18 - shrink, y),
            fill=(*CYAN, max(8, 42 - k * 4)),
            width=1,
        )

    # Landing ellipse
    lx = cx + 40 * np.sin(phase * 0.6)
    ly = terrain_base - 62
    ell_w = 88 - 28 * lock
    ell_h = 28 - 9 * lock

    d.ellipse(
        [lx - ell_w, ly - ell_h, lx + ell_w, ly + ell_h],
        outline=(*GREEN, int(90 + 110 * lock)),
        width=2,
    )
    d.text((lx - 50, ly + 34), "LANDING ELLIPSE", font=FONT_TINY, fill=(*GREEN, 165))

    # Hazards
    hazards = [
        (fx0 + 108, terrain_base - 70, "ROCK"),
        (fx0 + 315, terrain_base - 85, "SLOPE"),
        (fx0 + 380, terrain_base - 45, "HEAT"),
    ]

    for idx, (hx, hy, label) in enumerate(hazards):
        pulse = 0.55 + 0.45 * np.sin(phase * 4 + idx)
        col = RED if idx == 2 and heating > 0.25 else YELLOW
        a = int(150 * pulse)

        d.polygon(
            [(hx, hy - 12), (hx - 12, hy + 10), (hx + 12, hy + 10)],
            outline=(*col, a),
        )
        d.text((hx + 14, hy - 8), label, font=FONT_TINY, fill=(*col, int(165 * pulse)))

    # Craft
    craft_x = cx + 38 * np.sin(phase * 0.9) * (1 - 0.45 * lock)
    craft_y = fy0 + 82 + descent * 255

    d.line((craft_x, craft_y, lx, ly), fill=(*GREEN, 90), width=1)

    b = 22 + 5 * np.sin(phase * 5)
    draw_brackets(d, craft_x, craft_y, b, CYAN2)
    d.ellipse([craft_x - 4, craft_y - 4, craft_x + 4, craft_y + 4], fill=(*WHITE, 230))

    # Altitude ladder
    ladder_x = fx0 + 22

    for k in range(15):
        y = fy0 + 45 + k * 30
        tick = 24 if k % 2 == 0 else 14
        d.line((ladder_x, y, ladder_x + tick, y), fill=(*CYAN, 120), width=1)

        if k % 2 == 0:
            alt = int(12_000 * (1 - descent) - k * 360)
            d.text((ladder_x + 30, y - 7), f"{max(0, alt):05d}", font=FONT_TINY, fill=(*CYAN, 120))

    # Velocity marker
    vbar_x = fx1 - 38
    d.line((vbar_x, fy0 + 45, vbar_x, fy1 - 48), fill=(*CYAN, 65), width=1)
    vy = fy0 + 45 + (fy1 - fy0 - 95) * descent
    d.polygon([(vbar_x, vy), (vbar_x - 8, vy - 10), (vbar_x + 8, vy - 10)], fill=(*GREEN, 180))

    if heating > 0.05:
        a = int(130 * heating)
        d.rectangle([fx0, fy0, fx1, fy1], outline=(*ORANGE, a), width=3)
        d.text((fx0 + 18, fy1 - 30), "ATMOSPHERIC HEATING", font=FONT_TINY, fill=(*ORANGE, min(255, a + 40)))

    # Bottom telemetry panel
    px, py = 36, 640
    pw, ph = W - 72, 250

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 125, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "DESCENT SOLVER", font=FONT_SMALL, fill=(*CYAN2, 190))

    alt = 12000 * (1 - descent)
    vel = 3.8 - 2.4 * lock + 0.2 * np.sin(phase)
    heat = 18 + 72 * heating
    slope = 4.2 + 1.5 * np.sin(phase * 1.2)

    metrics = [
        ("ALT", f"{alt:06.0f} m"),
        ("VEL", f"{vel:04.2f} km/s"),
        ("HEAT", f"{heat:04.1f}%"),
        ("SLOPE", f"{slope:03.1f}°"),
        ("LOCK", f"{lock*100:04.1f}%"),
        ("STATE", "LOCK" if lock > 0.85 else "GUIDE"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 28
        col = GREEN if v == "LOCK" else CYAN2

        if k == "HEAT" and heat > 60:
            col = ORANGE

        d.text((px + 18, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 112, y), v, font=FONT_SMALL, fill=(*col, 220))

        for t in range(10):
            a = int(35 + 90 * np.sin(phase * 4 + m + t) ** 2)
            tx = px + 285 + t * 13
            d.line((tx, y + 8, tx + 8, y + 8), fill=(*CYAN, a), width=1)

    bx, by = px + 18, py + ph - 36

    for k in range(32):
        a = int(35 + 100 * np.sin(phase * 4 + k * 0.6) ** 2)
        h = 5 + 22 * np.sin(phase * 1.7 + k * 0.4) ** 2
        d.line((bx + k * 12, by, bx + k * 12, by - h), fill=(*CYAN2, a), width=2)

    d.text(
        (28, H - 34),
        "ALTITUDE LADDER • TERRAIN WIREFRAME • LANDING ELLIPSE",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 40, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 40, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.4))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "planetary_descent_hud_vertical.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/descent/planetary_descent_hud_vertical.gif


In [23]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/mission")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(8812)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)
FONT_BIG = load_font(22)


SYSTEMS = [
    ("NAV", "LOCK", GREEN),
    ("PROP", "NOMINAL", GREEN),
    ("LIFE", "STABLE", GREEN),
    ("COMMS", "SYNC", CYAN2),
    ("THERMAL", "HIGH", YELLOW),
    ("REACTOR", "WARN", ORANGE),
    ("SENSORS", "ACTIVE", GREEN),
    ("PAYLOAD", "READY", CYAN2),
    ("SHIELD", "LOW", RED),
]


TASKS = [
    "UPDATE TRAJECTORY SOLUTION",
    "VERIFY SENSOR ARRAY",
    "SYNC PAYLOAD BUFFER",
    "CHECK THERMAL LOOP",
    "PREPARE DATA BURST",
]


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    alert = np.exp(-0.5 * ((i - 108) / 12) ** 2)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((32, 10), "MISSION STATUS WALL", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 230, 34), fill=(*CYAN, 170), width=2)
    d.line((230, 34, 254, 48), fill=(*CYAN, 170), width=2)
    d.line((254, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Main frame
    x0, y0, x1, y1 = 42, 72, 918, 455
    d.rectangle([x0, y0, x1, y1], outline=(*CYAN, 85), width=1)
    d.line((x0, y0, x0 + 170, y0), fill=(*CYAN2, 190), width=3)

    # System tiles
    tile_w, tile_h = 180, 78
    gap = 14
    start_x, start_y = 62, 100

    for idx, (name, status, color) in enumerate(SYSTEMS):
        col = idx % 3
        row = idx // 3

        tx = start_x + col * (tile_w + gap)
        ty = start_y + row * (tile_h + gap)

        pulse = 0.55 + 0.45 * np.sin(phase * 3 + idx)
        aa = int(75 + 70 * pulse)

        if status in ("WARN", "HIGH", "LOW"):
            aa = int(95 + 105 * pulse)

        d.rectangle([tx, ty, tx + tile_w, ty + tile_h], outline=(*color, aa), width=1)
        d.rectangle([tx + 2, ty + 2, tx + tile_w - 2, ty + tile_h - 2], fill=(*CYAN, 12))

        d.text((tx + 14, ty + 12), name, font=FONT_SMALL, fill=(*CYAN2, 190))
        d.text((tx + 14, ty + 40), status, font=FONT_TINY, fill=(*color, 210))

        # Status dot
        d.ellipse([tx + tile_w - 30, ty + 16, tx + tile_w - 16, ty + 30], fill=(*color, int(180 * pulse)))

        # Micro activity ticks
        for k in range(8):
            a = int(35 + 90 * np.sin(phase * 5 + idx + k) ** 2)
            d.line((tx + 14 + k * 12, ty + tile_h - 12, tx + 21 + k * 12, ty + tile_h - 12), fill=(*CYAN, a), width=1)

    # Right summary panel
    px, py = 675, 100
    pw, ph = 220, 262

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 105, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "MISSION CORE", font=FONT_SMALL, fill=(*CYAN2, 190))

    readiness = 87 + 5 * np.sin(phase * 0.9) - 18 * alert
    risk = 14 + 52 * alert + 6 * np.sin(phase * 1.4) ** 2

    metrics = [
        ("READY", f"{readiness:04.1f}%"),
        ("RISK", f"{risk:04.1f}%"),
        ("TASKS", f"{len(TASKS):02d}"),
        ("ALERTS", "03" if alert > 0.2 else "01"),
        ("STATE", "CAUTION" if alert > 0.35 else "NOMINAL"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 56 + m * 38
        col = YELLOW if v == "CAUTION" else CYAN2
        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 94, y), v, font=FONT_SMALL, fill=(*col, 220))

    # Task queue
    qx, qy = 62, 390
    d.text((qx, qy - 20), "TASK QUEUE", font=FONT_TINY, fill=(*CYAN, 150))

    for idx, task in enumerate(TASKS):
        x = qx + idx * 168
        progress = (0.25 + 0.75 * ((i * 0.006 + idx * 0.13) % 1.0))
        col = GREEN if progress > 0.7 else CYAN2

        d.rectangle([x, qy, x + 145, qy + 36], outline=(*CYAN, 70), width=1)
        d.text((x + 8, qy + 7), task[:18], font=FONT_TINY, fill=(*CYAN2, 160))
        d.rectangle([x + 8, qy + 26, x + 137, qy + 31], outline=(*CYAN, 45), width=1)
        d.rectangle([x + 10, qy + 28, x + 10 + int(125 * progress), qy + 30], fill=(*col, 145))

    # Alert overlay
    if alert > 0.05:
        a = int(160 * alert)
        d.rectangle([x0, y0, x1, y1], outline=(*ORANGE, a), width=3)
        d.text((x0 + 18, y1 - 30), "SYSTEM LOAD EXCURSION DETECTED", font=FONT_TINY, fill=(*ORANGE, min(255, a + 40)))

    d.text(
        (32, H - 28),
        "MISSION READINESS • SYSTEM HEALTH MATRIX • TASK QUEUE • ALERT CONSOLIDATION",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.4))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "mission_status_wall.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/mission/mission_status_wall.gif


In [24]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/payload")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 540
FPS = 24
FRAMES = 180

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
ORANGE = (255, 160, 70)
RED = (255, 80, 110)
WHITE = (245, 250, 255)

rng = np.random.default_rng(9917)


def load_font(size):
    for path in [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)


PAYLOADS = [
    ("CAM-A", "IMAGING", GREEN),
    ("CAM-B", "STBY", CYAN2),
    ("SPEC", "ACQUIRE", GREEN),
    ("LIDAR", "SWEEP", CYAN2),
    ("ANTENNA", "SYNC", GREEN),
    ("CRYO", "LOW", YELLOW),
    ("BUFFER", "WRITE", CYAN2),
    ("RECORDER", "ACTIVE", GREEN),
]


def render_wave(d, x0, y0, w, phase, color):
    pts = []
    for x in range(w):
        yy = y0 + 7 * np.sin(x * 0.055 + phase) + 3 * np.sin(x * 0.18 - phase * 1.6)
        pts.append((x0 + x, yy))

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*color, 170), width=1)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES
    capture = min(1.0, max(0.0, (i - 18) / 125))
    warning = np.exp(-0.5 * ((i - 105) / 12) ** 2)

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    d.text((32, 10), "SCIENCE PAYLOAD CONTROL", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 255, 34), fill=(*CYAN, 170), width=2)
    d.line((255, 34, 278, 48), fill=(*CYAN, 170), width=2)
    d.line((278, 48, W - 42, 48), fill=(*CYAN, 75), width=1)

    # Instrument grid
    gx0, gy0 = 45, 74
    tile_w, tile_h = 190, 86
    gap_x, gap_y = 14, 14

    for idx, (name, state, color) in enumerate(PAYLOADS):
        col = idx % 3
        row = idx // 3

        x = gx0 + col * (tile_w + gap_x)
        y = gy0 + row * (tile_h + gap_y)

        pulse = 0.55 + 0.45 * np.sin(phase * 3 + idx)
        alpha = int(80 + 80 * pulse)

        d.rectangle([x, y, x + tile_w, y + tile_h], outline=(*color, alpha), width=1)
        d.line((x, y, x + 74, y), fill=(*CYAN2, 160), width=2)

        d.text((x + 14, y + 12), name, font=FONT_SMALL, fill=(*CYAN2, 190))
        d.text((x + 14, y + 38), state, font=FONT_TINY, fill=(*color, 210))

        level = 0.42 + 0.48 * np.sin(phase * (0.8 + idx * 0.07) + idx) ** 2
        if name == "CRYO":
            level = 0.25 + 0.28 * np.sin(phase * 1.3) ** 2 + 0.35 * warning

        d.rectangle([x + 14, y + 66, x + tile_w - 14, y + 73], outline=(*CYAN, 50), width=1)
        d.rectangle([x + 16, y + 68, x + 16 + int((tile_w - 32) * min(1, level)), y + 71], fill=(*color, 145))

        d.ellipse([x + tile_w - 28, y + 15, x + tile_w - 16, y + 27], fill=(*color, int(180 * pulse)))

    # Main observation panel
    px, py = 675, 82
    pw, ph = 235, 350

    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 105), width=1)
    d.line((px, py, px + 110, py), fill=(*CYAN2, 190), width=3)
    d.text((px + 16, py + 16), "OBSERVATION", font=FONT_SMALL, fill=(*CYAN2, 190))

    # Aperture / pointing target
    cx, cy = px + pw // 2, py + 116

    for r, a in [(78, 32), (52, 55), (28, 95)]:
        rr = r + 4 * np.sin(phase * 2 + r)
        d.ellipse([cx - rr, cy - rr, cx + rr, cy + rr], outline=(*CYAN, a), width=1)

    for k in range(16):
        ang = phase * 0.35 + k * 2 * np.pi / 16
        r0, r1 = 18, 82
        x0 = cx + np.cos(ang) * r0
        y0 = cy + np.sin(ang) * r0
        x1 = cx + np.cos(ang) * r1
        y1 = cy + np.sin(ang) * r1
        a = int(35 + 80 * np.sin(phase * 4 + k) ** 2)
        d.line((x0, y0, x1, y1), fill=(*CYAN2, a), width=1)

    d.ellipse([cx - 7, cy - 7, cx + 7, cy + 7], fill=(*WHITE, 220))

    metrics = [
        ("POINT", f"{91 + 7*np.sin(phase):04.1f}%"),
        ("EXPOS", f"{capture*100:04.1f}%"),
        ("DATA", f"{3.2 + 4.8*capture:04.1f} GB"),
        ("TEMP", f"{82 + 12*warning:04.1f} K"),
        ("LINK", "SYNC"),
        ("STATE", "WARN" if warning > 0.35 else "ACTIVE"),
    ]

    for m, (k, v) in enumerate(metrics):
        y = py + 210 + m * 29
        col = ORANGE if v == "WARN" else (GREEN if v in ("SYNC", "ACTIVE") else CYAN2)

        d.text((px + 16, y), k, font=FONT_TINY, fill=(*CYAN, 150))
        d.text((px + 92, y), v, font=FONT_SMALL, fill=(*col, 220))

    # Waveform / data stream
    wx0, wy0 = 60, 384
    wx1, wy1 = 630, 438

    d.rectangle([wx0, wy0, wx1, wy1], outline=(*CYAN, 70), width=1)
    render_wave(d, wx0 + 14, wy0 + 26, wx1 - wx0 - 28, phase * 2.1, CYAN2)

    # Data packet bars
    bx, by = 60, 466
    d.text((bx, by - 22), "PAYLOAD DATA BUFFER", font=FONT_TINY, fill=(*CYAN, 150))

    for k in range(54):
        a = int(35 + 100 * np.sin(phase * 4 + k * 0.45) ** 2)
        h = 6 + 28 * np.sin(phase * 1.6 + k * 0.22) ** 2
        col = GREEN if k < int(54 * capture) else CYAN
        d.line((bx + k * 10, by, bx + k * 10, by - h), fill=(*col, a), width=2)

    if warning > 0.05:
        a = int(150 * warning)
        d.rectangle([gx0 - 8, gy0 - 8, gx0 + 3 * tile_w + 2 * gap_x + 8, gy0 + 3 * tile_h + 2 * gap_y + 8], outline=(*ORANGE, a), width=3)
        d.text((gx0 + 18, gy0 + 3 * tile_h + 2 * gap_y + 18), "CRYO LOOP TEMPERATURE EXCURSION", font=FONT_TINY, fill=(*ORANGE, min(255, a + 40)))

    d.text(
        (32, H - 28),
        "SCIENCE PAYLOAD ARRAY • OBSERVATION SEQUENCE • DATA BUFFER • THERMAL CONTROL",
        font=FONT_TINY,
        fill=(*CYAN, 130),
    )

    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(1.4))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "science_payload_control.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/payload/science_payload_control.gif
